In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!git clone https://github.com/TebogoSeopa20/Honours-Research-Project.git
%cd /kaggle/working/Honours-Research-Project

Cloning into 'Honours-Research-Project'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 323 (delta 170), reused 246 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (323/323), 311.61 KiB | 2.23 MiB/s, done.
Resolving deltas: 100% (170/170), done.
/kaggle/working/Honours-Research-Project


In [2]:
!pip install -e ".[test]"
!python -m pytest tests/ -q --ignore=tests/test_label_masking.py

Obtaining file:///kaggle/working/Honours-Research-Project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.2 MB/s eta 0:00:00:00:0100:01
  Building editable for misinfodet (pyproject.toml) ... done
  Created wheel for misinfodet: filename=misinfodet-0.1.0-0.editable-py3-none-any.whl size=1617 sha256=54228bf4db64af8d7cbdbe3871e0fc0d5a76c8c589fa2e86fd9b12487e4065c8
  Stored in directory: /tmp/pip-ephem-wheel-cache-g6efam3n/wheels/19/a6/60/20c19df9f92a41b07433d31748a9c982c47838f9655171f3e4
Successfully built misinfodet
.....................                                                    [100%]
21 passed in 1.42s


In [2]:
!git pull

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

model_id = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded successfully")

In [ ]:
!mkdir -p /kaggle/working/data/cosmos
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/train_data.json /kaggle/working/data/cosmos/train_data.json
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/val_data.json /kaggle/working/data/cosmos/val_data.json
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/test_data.json /kaggle/working/data/cosmos/test_data.json
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_train/train /kaggle/working/data/cosmos/train
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_val/val /kaggle/working/data/cosmos/val
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_test_clean/test /kaggle/working/data/cosmos/test
!sed -i 's|cosmos_dir:.*|cosmos_dir: /kaggle/working/data/cosmos|' configs/base.yaml
!python scripts/prepare_cosmos.py --config configs/base.yaml

In [ ]:
!mkdir -p outputs/checkpoints/stage1_best_local
!cp /kaggle/input/datasets/tebogoseopa20/stage1-best-v2/* outputs/checkpoints/stage1_best_local/

In [ ]:
!sed -i 's|cosmos_dir:.*|cosmos_dir: /kaggle/working/data/cosmos|' configs/base.yaml
!python scripts/prepare_cosmos.py --config configs/base.yaml

In [ ]:
!ls -la data/cosmos_labeled_train.jsonl

In [ ]:
!ls outputs/checkpoints/stage1_best_local/

In [ ]:
!sed -i 's|max_samples:.*|max_samples: 20|' configs/stage2_ooc_finetune.yaml
!python scripts/train_stage2.py --config configs/stage2_ooc_finetune.yaml

In [ ]:
!sed -i 's|max_samples:.*|max_samples: null|' configs/stage2_ooc_finetune.yaml
!python scripts/train_stage2.py --config configs/stage2_ooc_finetune.yaml

In [ ]:
!git pull
!python scripts/prepare_cosmos.py --config configs/base.yaml

In [ ]:
!python scripts/train_stage2.py --config configs/stage2_ooc_finetune.yaml

In [ ]:
!kaggle datasets files -d tebogoseopa20/stage2-checkpoints-best

In [ ]:
!grep -n "kaggle_checkpoint_dataset}-best" src/misinfodet/models/training.py

In [ ]:
!kaggle datasets init -p outputs/checkpoints/stage2_best
!python -c "import json; json.dump({'title':'stage2 best checkpoint','id':'tebogoseopa20/stage2-best','licenses':[{'name':'CC0-1.0'}]}, open('outputs/checkpoints/stage2_best/dataset-metadata.json','w'))"
!kaggle datasets create -p outputs/checkpoints/stage2_best

In [ ]:
!kaggle datasets files -d tebogoseopa20/stage2-best

In [ ]:
!python scripts/run_inference.py --help


In [ ]:
!git pull
!ls outputs/checkpoints/stage1_best_local/
!ls outputs/checkpoints/stage2_best/

In [ ]:
!python scripts/run_inference.py --config configs/stage2_eval.yaml --dataset cosmos --split test

In [ ]:
!python scripts/evaluate.py --help

In [ ]:
!python scripts/evaluate.py --predictions outputs/predictions/stage2_eval_cosmos_test.jsonl

In [ ]:
!python scripts/run_baseline.py --config configs/base.yaml --dataset cosmos --split test
!python scripts/evaluate.py --predictions outputs/predictions/baseline_cosmos_test.jsonl

In [ ]:
!git pull
!grep -c "kaggle_checkpoint_dataset}-best" src/misinfodet/models/training.py

In [ ]:
!grep -E "epochs:|kaggle_checkpoint_dataset:" configs/stage2_ooc_finetune.yaml

In [ ]:
!python scripts/train_stage2.py --config configs/stage2_ooc_finetune.yaml

In [ ]:
!mkdir -p outputs/checkpoints/stage2_best_3ep_local
!cp /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-3ep-best/* outputs/checkpoints/stage2_best_3ep_local/
!ls outputs/checkpoints/stage2_best_3ep_local/

In [ ]:
!ls /kaggle/input/datasets/tebogoseopa20/

In [ ]:
!kaggle datasets files -d tebogoseopa20/stage2-checkpoints-3ep

In [ ]:
!kaggle datasets init -p outputs/checkpoints/stage2
!python -c "import json; json.dump({'title':'stage2 checkpoints 3ep manual','id':'tebogoseopa20/stage2-checkpoints-3ep-manual','licenses':[{'name':'CC0-1.0'}]}, open('outputs/checkpoints/stage2/dataset-metadata.json','w'))"
!kaggle datasets create -p outputs/checkpoints/stage2

In [ ]:
!kaggle datasets init -p outputs/checkpoints/stage2_best
!python -c "import json; json.dump({'title':'stage2 best 3ep manual','id':'tebogoseopa20/stage2-best-3ep-manual','licenses':[{'name':'CC0-1.0'}]}, open('outputs/checkpoints/stage2_best/dataset-metadata.json','w'))"
!kaggle datasets create -p outputs/checkpoints/stage2_best

In [ ]:
!kaggle datasets files -d tebogoseopa20/stage2-checkpoints-3ep-manual
!kaggle datasets files -d tebogoseopa20/stage2-best-3ep-manual

In [ ]:
!mkdir -p outputs/checkpoints/stage2_best_3ep_local
!cp outputs/checkpoints/stage2_best/* outputs/checkpoints/stage2_best_3ep_local/
!ls outputs/checkpoints/stage2_best_3ep_local/

In [ ]:
!python scripts/run_inference.py --config configs/stage2_eval_3ep.yaml --dataset cosmos --split test
!python scripts/evaluate.py --predictions outputs/predictions/stage2_eval_3ep_cosmos_test.jsonl

In [48]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.93 KiB | 2.93 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   0e05b03..67ace16  main       -> origin/main
Updating 0e05b03..67ace16
Fast-forward
 scripts/fusion_calibration.py | 166 ++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 166 insertions(+)
 create mode 100644 scripts/fusion_calibration.py


In [ ]:
!python scripts/run_inference.py --config configs/stage2_eval_3ep.yaml --dataset cosmos --split test
!python scripts/evaluate.py --predictions outputs/predictions/stage2_eval_3ep_cosmos_test.jsonl

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["SEARCH_API_KEY"] = UserSecretsClient().get_secret("SEARCH_API_KEY")

In [ ]:
print("Key loaded:", bool(os.environ.get("SEARCH_API_KEY")))
print("Key length:", len(os.environ.get("SEARCH_API_KEY", "")))

In [ ]:
!sed -i 's|use_retrieval:.*|use_retrieval: true|; s|use_reranking:.*|use_reranking: true|' configs/stage2_eval_3ep.yaml
!echo 'retrieval_top_k: 8' >> configs/stage2_eval_3ep.yaml
!echo 'rerank_keep_k: 3' >> configs/stage2_eval_3ep.yaml
!echo 'search_provider: serper' >> configs/stage2_eval_3ep.yaml
!echo 'search_api_key_env: SEARCH_API_KEY' >> configs/stage2_eval_3ep.yaml
!cat configs/stage2_eval_3ep.yaml

In [ ]:
!python scripts/run_inference.py --config configs/stage2_eval_3ep.yaml --dataset cosmos --split test

In [ ]:
!head -1 outputs/predictions/stage2_eval_3ep_cosmos_test.jsonl

In [ ]:
import requests, os
resp = requests.post(
    "https://google.serper.dev/search",
    headers={"X-API-KEY": os.environ["SEARCH_API_KEY"], "Content-Type": "application/json"},
    json={"q": "\"London Brexit march December 2018\""}
)
print("WITH literal quotes:", len(resp.json().get("organic", [])), "results")

resp2 = requests.post(
    "https://google.serper.dev/search",
    headers={"X-API-KEY": os.environ["SEARCH_API_KEY"], "Content-Type": "application/json"},
    json={"q": "London Brexit march December 2018"}
)
print("WITHOUT quotes:", len(resp2.json().get("organic", [])), "results")

In [ ]:
!git pull
!python scripts/run_inference.py --config configs/stage2_eval_3ep.yaml --dataset cosmos --split test

In [ ]:
!head -1 outputs/predictions/stage2_eval_3ep_cosmos_test.jsonl

In [ ]:
!python scripts/evaluate.py --predictions outputs/predictions/stage2_eval_3ep_cosmos_test.jsonl

In [ ]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml --debug

In [ ]:
!mkdir -p outputs/checkpoints/stage1_best_local
!cp /kaggle/input/datasets/tebogoseopa20/stage1-best-v2/* outputs/checkpoints/stage1_best_local/

!mkdir -p outputs/checkpoints/stage2_best_3ep_local
!cp /kaggle/input/datasets/tebogoseopa20/stage2-best-3ep-manual/* outputs/checkpoints/stage2_best_3ep_local/

In [ ]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml --debug

In [ ]:
!git pull
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml --debug

In [ ]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

In [ ]:
print("still alive")

In [ ]:
!nvidia-smi

In [ ]:
!git pull

In [ ]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

In [ ]:
!head -3 outputs/predictions/stage2_eval_3ep_cosmos_test.jsonl

In [ ]:
from misinfodet.models.model_factory import load_llava, generate
from misinfodet.pipeline.prompts import PAIRED_VERDICT_PROMPT
from misinfodet.utils import Config, read_jsonl
from pathlib import Path
from PIL import Image

cfg = Config.from_yaml("configs/stage2_eval_3ep.yaml")
model, processor = load_llava(cfg, trainable=False)

test_records = read_jsonl(Path(cfg.data_dir) / "cosmos_labeled_test.jsonl")
for rec in test_records[:3]:
    image = Image.open(rec["image_path"]).convert("RGB")
    prompt = PAIRED_VERDICT_PROMPT.format(caption1=rec["caption1"], caption2=rec["caption2"])
    raw = generate(model, processor, image, prompt, cfg)
    print(f"gold_label={rec['label']}")
    print(f"raw_response: {raw!r}")
    print("---")

In [2]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [3]:
from misinfodet.models.model_factory import load_llava, generate
from misinfodet.pipeline.prompts import PAIRED_VERDICT_PROMPT
from misinfodet.utils import Config, read_jsonl
from pathlib import Path
from PIL import Image

cfg = Config.from_yaml("configs/stage2_eval_3ep.yaml")
model, processor = load_llava(cfg, trainable=False)

test_records = read_jsonl(Path(cfg.data_dir) / "cosmos_labeled_test.jsonl")
for rec in test_records[:3]:
    image = Image.open(rec["image_path"]).convert("RGB")
    prompt = PAIRED_VERDICT_PROMPT.format(caption1=rec["caption1"], caption2=rec["caption2"])
    raw = generate(model, processor, image, prompt, cfg)
    print(f"gold_label={rec['label']}")
    print(f"raw_response: {raw!r}")
    print("---")

[2026-09-20 21:22:24,946] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

[2026-09-20 21:22:36,115] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.1.self_attn.k_pro

gold_label=0
raw_response: 'Out-of-context'
---
gold_label=1
raw_response: 'Consistent.'
---
gold_label=1
raw_response: 'Out-of-context'
---


In [4]:
!git pull
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.01 KiB | 517.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   2e941bc..dedd324  main       -> origin/main
Updating 2e941bc..dedd324
Fast-forward
 src/misinfodet/models/model_factory.py | 36 +++++++++++++++++-----------------
 1 file changed, 18 insertions(+), 18 deletions(-)
Loading model...
[2026-09-20 21:28:28,939] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 31694.99it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|███████████████████████| 686/686 [00:04<00:00, 139.64it/s]
[2026-09-20 21:28:39,625] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/c

In [5]:
!git pull
!python scripts/diagnose_gap.py --config configs/stage2_eval_3ep.yaml --n 25

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.92 KiB | 1.92 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   dedd324..1bc3078  main       -> origin/main
Updating dedd324..1bc3078
Fast-forward
 scripts/diagnose_gap.py | 94 +++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 94 insertions(+)
 create mode 100644 scripts/diagnose_gap.py
Loading model...
[2026-09-20 21:46:35,224] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 43539.49it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|███████████████████████| 686/686 [00:05<00:00, 136.27it/s]
[2026-09-20 21:46:46,268] INFO misinfodet.models.model_factory: Applying stage1 

In [6]:
!git pull
!python scripts/diagnose_gap.py --config configs/stage2_eval_3ep.yaml --n 25

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.12 KiB | 572.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   1bc3078..5100315  main       -> origin/main
Updating 1bc3078..5100315
Fast-forward
 src/misinfodet/models/model_factory.py | 40 ++++++++++++++++++++--------------
 1 file changed, 24 insertions(+), 16 deletions(-)
Loading model...
[2026-09-20 21:50:54,491] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 50131.12it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|███████████████████████| 686/686 [00:05<00:00, 134.98it/s]
[2026-09-20 21:51:05,641] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/c

In [7]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

Loading model...
[2026-09-20 21:56:02,845] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 6890.97it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|███████████████████████| 686/686 [00:05<00:00, 134.31it/s]
[2026-09-20 21:56:13,911] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 

In [1]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="liuxuannan/MMFakeBench", filename="MMFakeBench_val.json",
                repo_type="dataset", local_dir="/kaggle/working/mmfakebench", token=os.environ["HF_TOKEN"])
hf_hub_download(repo_id="liuxuannan/MMFakeBench", filename="MMFakeBench_val.zip",
                repo_type="dataset", local_dir="/kaggle/working/mmfakebench", token=os.environ["HF_TOKEN"])

MMFakeBench_val.json: 0.00B [00:00, ?B/s]

MMFakeBench_val.zip:   0%|          | 0.00/646M [00:00<?, ?B/s]

'/kaggle/working/mmfakebench/MMFakeBench_val.zip'

In [2]:
!cd /kaggle/working/mmfakebench && unzip -q MMFakeBench_val.zip
!ls /kaggle/working/mmfakebench/

MMFakeBench_val  MMFakeBench_val.json  MMFakeBench_val.zip


In [3]:
!ls /kaggle/working/mmfakebench/MMFakeBench_val/

fake  real  source


In [4]:
import json
with open("/kaggle/working/mmfakebench/MMFakeBench_val.json") as f:
    # It's likely JSONL given COSMOS's precedent, but check first
    first_line = f.readline()
print(repr(first_line[:500]))

'[\n'


In [5]:
import json
with open("/kaggle/working/mmfakebench/MMFakeBench_val.json") as f:
    data = json.load(f)

print(f"Total records: {len(data)}")
print(json.dumps(data[0], indent=2))

Total records: 1000
{
  "text": "Jake Davis who has been released from a young offender institution first appeared in court in 2011",
  "image_path": "/real/bbc_val_50/BBC_val_0.png",
  "text_source": "VisualNews",
  "image_source": "VisualNews",
  "gt_answers": "True",
  "fake_cls": "original"
}


In [6]:
fake_examples = [r for r in data if r["gt_answers"] == "False"]
print(f"Fake examples: {len(fake_examples)}")
print(json.dumps(fake_examples[0], indent=2))

Fake examples: 0


IndexError: list index out of range

In [7]:
labels = set(r["gt_answers"] for r in data)
print(labels)

{'True', 'Fake'}


In [8]:
fake_examples = [r for r in data if r["gt_answers"] == "Fake"]
print(f"Fake examples: {len(fake_examples)}")
print(json.dumps(fake_examples[0], indent=2))

Fake examples: 700
{
  "text": "The Chrysler Building has yet to be surpassed in height.",
  "image_path": "/fake/fever_AI_val_100/fever_val_AI_1.png",
  "text_source": "Fever",
  "image_source": "AI-generated Image",
  "gt_answers": "Fake",
  "fake_cls": "textual_veracity_distortion"
}


In [9]:
!ls /kaggle/working/mmfakebench/

MMFakeBench_val  MMFakeBench_val.json  MMFakeBench_val.zip


In [10]:
from huggingface_hub import hf_hub_download
import os

hf_hub_download(repo_id="liuxuannan/MMFakeBench", filename="MMFakeBench_test.json",
                repo_type="dataset", local_dir="/kaggle/working/mmfakebench", token=os.environ["HF_TOKEN"])
hf_hub_download(repo_id="liuxuannan/MMFakeBench", filename="MMFakeBench_test.zip",
                repo_type="dataset", local_dir="/kaggle/working/mmfakebench", token=os.environ["HF_TOKEN"])

MMFakeBench_test.json: 0.00B [00:00, ?B/s]

MMFakeBench_test.zip:   0%|          | 0.00/6.81G [00:00<?, ?B/s]

'/kaggle/working/mmfakebench/MMFakeBench_test.zip'

In [11]:
!cd /kaggle/working/mmfakebench && unzip -q MMFakeBench_test.zip
!ls /kaggle/working/mmfakebench/

MMFakeBench_test       MMFakeBench_test.zip  MMFakeBench_val.json
MMFakeBench_test.json  MMFakeBench_val	     MMFakeBench_val.zip


In [12]:
!kaggle datasets init -p /kaggle/working/mmfakebench


Data package template written to: /kaggle/working/mmfakebench/dataset-metadata.json


In [14]:
import json
json.dump(
    {"title": "mmfakebench", "id": "tebogoseopa20/mmfakebench", "licenses": [{"name": "CC0-1.0"}]},
    open("/kaggle/working/mmfakebench/dataset-metadata.json", "w")
)

In [15]:
!kaggle datasets create -p /kaggle/working/mmfakebench --dir-mode zip

Starting upload for file MMFakeBench_val.zip
100%|████████████████████████████████████████| 616M/616M [00:06<00:00, 95.4MB/s]
Upload successful: MMFakeBench_val.zip (616MB)
Starting upload for file MMFakeBench_test.json
100%|██████████████████████████████████████| 2.99M/2.99M [00:00<00:00, 4.45MB/s]
Upload successful: MMFakeBench_test.json (3MB)
Starting upload for file MMFakeBench_val.json
100%|█████████████████████████████████████████| 301k/301k [00:00<00:00, 516kB/s]
Upload successful: MMFakeBench_val.json (301KB)
Starting upload for file MMFakeBench_test.zip
100%|██████████████████████████████████████| 6.34G/6.34G [01:34<00:00, 71.9MB/s]
Upload successful: MMFakeBench_test.zip (6GB)
Starting upload for file MMFakeBench_val.zip
100%|████████████████████████████████████████| 616M/616M [00:08<00:00, 74.6MB/s]
Upload successful: MMFakeBench_val.zip (616MB)
Starting upload for file MMFakeBench_test.zip
100%|██████████████████████████████████████| 6.34G/6.34G [01:21<00:00, 83.7MB/s]
Uplo

In [16]:
!rm /kaggle/working/mmfakebench/MMFakeBench_val.zip
!rm /kaggle/working/mmfakebench/MMFakeBench_test.zip
!kaggle datasets create -p /kaggle/working/mmfakebench --dir-mode zip

Starting upload for file MMFakeBench_test.json
100%|██████████████████████████████████████| 2.99M/2.99M [00:00<00:00, 4.47MB/s]
Upload successful: MMFakeBench_test.json (3MB)
Starting upload for file MMFakeBench_val.json
100%|█████████████████████████████████████████| 301k/301k [00:00<00:00, 538kB/s]
Upload successful: MMFakeBench_val.json (301KB)
Starting upload for file MMFakeBench_test.zip
100%|██████████████████████████████████████| 6.34G/6.34G [01:20<00:00, 84.1MB/s]
Upload successful: MMFakeBench_test.zip (6GB)
Starting upload for file MMFakeBench_val.zip
100%|████████████████████████████████████████| 616M/616M [00:07<00:00, 81.9MB/s]
Upload successful: MMFakeBench_val.zip (616MB)
Starting upload for file .cache.zip
100%|██████████████████████████████████████| 1.63k/1.63k [00:00<00:00, 2.67kB/s]
Upload successful: .cache.zip (2KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/tebogoseopa20/mmfakebench


In [17]:
!kaggle datasets files -d tebogoseopa20/mmfakebench

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/ListDatasetFiles


In [18]:
!kaggle datasets files -d tebogoseopa20/mmfakebench

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/ListDatasetFiles


In [19]:
!kaggle datasets create -p /kaggle/working/mmfakebench --dir-mode zip

Dataset creation error: The requested title "mmfakebench" is already in use by a dataset. Please choose another title.


In [20]:
import json
json.dump(
    {"title": "mmfakebench data", "id": "tebogoseopa20/mmfakebench-data", "licenses": [{"name": "CC0-1.0"}]},
    open("/kaggle/working/mmfakebench/dataset-metadata.json", "w")
)
!kaggle datasets create -p /kaggle/working/mmfakebench --dir-mode zip

Starting upload for file MMFakeBench_test.json
100%|██████████████████████████████████████| 2.99M/2.99M [00:00<00:00, 4.56MB/s]
Upload successful: MMFakeBench_test.json (3MB)
Starting upload for file MMFakeBench_val.json
100%|█████████████████████████████████████████| 301k/301k [00:00<00:00, 489kB/s]
Upload successful: MMFakeBench_val.json (301KB)
^C
User cancelled operation


In [3]:
!git pull
!grep -c "resume_from_adapter" src/misinfodet/utils.py src/misinfodet/models/model_factory.py scripts/train_stage2.py

Already up to date.
src/misinfodet/utils.py:1
src/misinfodet/models/model_factory.py:2
scripts/train_stage2.py:1


In [4]:
!mkdir -p outputs/checkpoints
!cp -r /kaggle/input/datasets/tebogoseopa20/stage1-best-v2 outputs/checkpoints/stage1_best_local
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-best-3ep-manual outputs/checkpoints/stage2_resume_base

In [5]:
!ls outputs/checkpoints/stage1_best_local
!ls outputs/checkpoints/stage2_resume_base

adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json
adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json


In [7]:
!mkdir -p /kaggle/working/data/cosmos
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/train_data.json /kaggle/working/data/cosmos/train_data.json
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/val_data.json /kaggle/working/data/cosmos/val_data.json
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/test_data.json /kaggle/working/data/cosmos/test_data.json
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_train/train /kaggle/working/data/cosmos/train
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_val/val /kaggle/working/data/cosmos/val
!ln -s /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_test_clean/test /kaggle/working/data/cosmos/test
!sed -i 's|cosmos_dir:.*|cosmos_dir: /kaggle/working/data/cosmos|' configs/base.yaml
!python scripts/prepare_cosmos.py --config configs/base.yaml

[2026-09-26 10:09:50,548] INFO misinfodet.data.cosmos: COSMOS train: min_entity_freq=3 (spaCy PERSON) kept 83081/161754 images with a repeated-person caption
[2026-09-26 10:09:50,732] INFO misinfodet.data.cosmos: COSMOS train: wrote 12000 unlabeled records to data/cosmos_train.jsonl
[2026-09-26 10:12:12,828] INFO misinfodet.data.cosmos: COSMOS val: min_entity_freq=3 (spaCy PERSON) kept 18550/41006 images with a repeated-person caption
[2026-09-26 10:12:12,980] INFO misinfodet.data.cosmos: COSMOS val: wrote 12000 unlabeled records to data/cosmos_val.jsonl
[2026-09-26 10:12:17,871] INFO misinfodet.data.cosmos: COSMOS test: skipped 1 records with missing image files (known artifact, see docstring)
[2026-09-26 10:12:18,676] INFO misinfodet.data.cosmos: COSMOS labeled split (seed=42): 1189 train / 254 val / 256 test
train: 12000 records
val: 12000 records
labeled_train: 1189 records
labeled_val: 254 records
labeled_test: 256 records


In [ ]:
!python scripts/train_stage2.py --config configs/stage2_resume.yaml

Stage 2: 1189 OOC fine-tuning examples
Stage 2: 254 held-out validation examples
Stage 2: RESUMING training from existing adapter: outputs/checkpoints/stage2_resume_base
Stage 2: running 2 additional epoch(s) on top of it
[2026-09-26 10:15:02,737] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 4.49MB/s]
model.safetensors.index.json: 70.1kB [00:00, 134MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [01:06<00:00, 22.31s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:10<00:00, 199MB/s]
Loading weights: 100%|█| 686/686 [00:12<00:00, 54.36it/s, Materializing param=mo
preprocessor_config.json: 100%|█████████████████| 505/505 [00:00<00:00, 369kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce 

In [11]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 593 bytes | 593.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   8ee88a7..b2746f8  main       -> origin/main
Updating 8ee88a7..b2746f8
Fast-forward
 configs/stage2_eval_resumed.yaml | 15 +++++++++++++++
 1 file changed, 15 insertions(+)
 create mode 100644 configs/stage2_eval_resumed.yaml


In [12]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_resumed.yaml --debug

Loading model...
[2026-09-26 13:32:36,181] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 9475.08it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:05<00:00, 129.54it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 13:32:46,417] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
[2026-09-26 13:3

In [13]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_resumed.yaml

Loading model...
[2026-09-26 13:34:19,008] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 41255.45it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 154.81it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 13:34:27,710] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
[2026-09-26 13:3

In [14]:
!mkdir -p outputs/checkpoints/stage2_best_3ep_local
!cp /kaggle/input/datasets/tebogoseopa20/stage2-best-3ep-manual/* outputs/checkpoints/stage2_best_3ep_local/
!ls outputs/checkpoints/stage2_best_3ep_local

adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json


In [15]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

Loading model...
[2026-09-26 13:47:00,678] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 13486.51it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 148.72it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 13:47:09,618] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
[2026-09-26 13:4

In [16]:
!git pull
!python scripts/calibrate_threshold.py --config configs/diag_stage1_only.yaml

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 605 bytes | 605.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   b2746f8..b40dc48  main       -> origin/main
Updating b2746f8..b40dc48
Fast-forward
 configs/diag_stage1_only.yaml | 15 +++++++++++++++
 1 file changed, 15 insertions(+)
 create mode 100644 configs/diag_stage1_only.yaml
Loading model...
[2026-09-26 13:59:52,720] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 25944.15it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 146.53it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even

In [20]:
!git pull

Already up to date.


In [21]:
!grep -n "Attach every configured prior-stage adapter FIRST" src/misinfodet/models/model_factory.py

57:    # Attach every configured prior-stage adapter FIRST, unmerged, in order —


In [22]:
!mkdir -p outputs/checkpoints/stage1_best_local outputs/checkpoints/stage2_best_3ep_local outputs/checkpoints/stage2_best
!cp -r /kaggle/input/stage1-best-v2/* outputs/checkpoints/stage1_best_local/
!cp -r /kaggle/input/stage2-checkpoints-3ep-manual/* outputs/checkpoints/stage2_best_3ep_local/
!cp -r /kaggle/input/stage2-checkpoints-resumed-best/* outputs/checkpoints/stage2_best/

cp: cannot stat '/kaggle/input/stage1-best-v2/*': No such file or directory
cp: cannot stat '/kaggle/input/stage2-checkpoints-3ep-manual/*': No such file or directory
cp: cannot stat '/kaggle/input/stage2-checkpoints-resumed-best/*': No such file or directory


In [23]:
!ls /kaggle/input/

datasets


In [24]:
!ls /kaggle/input/datasets/

tebogoseopa20


In [25]:
!ls /kaggle/input/datasets/tebogoseopa20/

cosmos-dataset	stage2-best		stage2-checkpoints-resumed-best
stage1-best-v2	stage2-best-3ep-manual


In [26]:
!mkdir -p outputs/checkpoints/stage1_best_local outputs/checkpoints/stage2_best_3ep_local outputs/checkpoints/stage2_best
!cp -r /kaggle/input/datasets/tebogoseopa20/stage1-best-v2/* outputs/checkpoints/stage1_best_local/
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-best-3ep-manual/* outputs/checkpoints/stage2_best_3ep_local/
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-resumed-best/* outputs/checkpoints/stage2_best/

In [27]:
!ls data/cosmos/ 2>/dev/null; ls data/ 2>/dev/null

cosmos_labeled_test.jsonl   cosmos_labeled_val.jsonl  cosmos_val.jsonl
cosmos_labeled_train.jsonl  cosmos_train.jsonl


In [28]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

Loading model...
[2026-09-26 14:30:00,485] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 34855.71it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 151.38it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 14:30:09,360] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-26 14:30:09,933] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_3ep_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:

In [29]:
!git pull
!grep -n "merge_adapter" src/misinfodet/models/model_factory.py

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 872 bytes | 436.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   8720dd0..e85b362  main       -> origin/main
Updating 8720dd0..e85b362
Fast-forward
 src/misinfodet/models/model_factory.py | 10 ++++++++--
 1 file changed, 8 insertions(+), 2 deletions(-)
71:        # merge_adapter() merges every nested adapter's LoRA deltas into its
76:        # merge_adapter() walks every submodule regardless of nesting depth,
78:        model.merge_adapter()


In [30]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

Loading model...
[2026-09-26 14:38:03,731] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 3876.44it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 155.34it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 14:38:12,547] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-26 14:38:13,113] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_3ep_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:

In [31]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_resumed.yaml

Loading model...
[2026-09-26 14:47:08,497] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 24966.10it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 153.13it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 14:47:17,398] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-26 14:47:17,983] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserW

In [32]:
!git log --oneline -- configs/stage2_eval_3ep.yaml

41f020b training stage 2


In [33]:
!git log --oneline -- scripts/calibrate_threshold.py

2e941bc training stage 2
028b085 training stage 2


In [34]:
!git diff 028b085 2e941bc -- scripts/calibrate_threshold.py

diff --git a/scripts/calibrate_threshold.py b/scripts/calibrate_threshold.py
index 53e81c3..b95dbaf 100644
--- a/scripts/calibrate_threshold.py
+++ b/scripts/calibrate_threshold.py
@@ -28,14 +28,25 @@ from misinfodet.pipeline.prompts import PAIRED_VERDICT_PROMPT
 from misinfodet.utils import Config, read_jsonl
 
 
-def score_dataset(model, processor, records, cfg):
-    """Returns [(score, gold_label), ...] — score = P(out-of-context)."""
+def score_dataset(model, processor, records, cfg, label: str = ""):
+    """Returns [(score, gold_label), ...] — score = P(out-of-context).
+
+    Frees GPU memory every 20 examples and prints progress — a silent
+    OOM crash partway through 500+ forward passes (confirmed: process
+    disappeared entirely, no traceback, GPU memory back to 0MiB) is the
+    real failure mode here, not a slow-but-alive computation.
+    """
+    import torch
+
     results = []
-    for rec in records:
+    for i, rec in enumerate(records):
         image = Image.op

In [35]:
!git log --oneline -- src/misinfodet/models/model_factory.py

e85b362 (HEAD -> main, origin/main, origin/HEAD) Fix crash on nested PeftModel merge: use merge_adapter() instead of merge_and_unload()
8720dd0 Fix stage2 adapter no-op bug: attach adapters before merging at inference
8ee88a7 worked on improving stage 2 training
5100315 training stage 2
dedd324 training stage 2
69756c4 training stage 2
028b085 training stage 2
af33aa4 working on research


In [36]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_3ep.yaml

Loading model...
[2026-09-26 15:10:21,707] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 45425.68it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 152.79it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-26 15:10:30,808] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-26 15:10:31,371] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_3ep_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:

In [39]:
!grep -A3 "lora_target_modules" configs/stage2_ooc_finetune.yaml configs/stage1_*.yaml 2>/dev/null

In [40]:
!ls configs/ | grep -i stage1

diag_stage1_only.yaml
stage1_entity_alignment.yaml


In [41]:
!ls configs/ | grep -i stage2

stage2_eval_3ep.yaml
stage2_eval_resumed.yaml
stage2_eval.yaml
stage2_ooc_finetune.yaml
stage2_resume.yaml


In [43]:
!grep -rl "lora_target_modules" configs/

In [3]:
!cat configs/stage2_ooc_finetune.yaml

run_name: stage2_ooc_finetune_3ep
seed: 42
base_model: llava-hf/llava-1.5-7b-hf
load_in_4bit: true
stage1_adapter: outputs/checkpoints/stage1_best_local
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
epochs: 3
batch_size: 4
grad_accum: 4
lr: 1.0e-4
warmup_ratio: 0.03
wandb: false
max_samples: null
save_every_steps: 20
kaggle_checkpoint_dataset: tebogoseopa20/stage2-checkpoints-3ep

In [4]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 868 bytes | 868.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   e85b362..a447483  main       -> origin/main
Updating e85b362..a447483
Fast-forward
 configs/stage2_eval_lmonly.yaml         |  8 ++++++++
 configs/stage2_ooc_finetune_lmonly.yaml | 18 ++++++++++++++++++
 2 files changed, 26 insertions(+)
 create mode 100644 configs/stage2_eval_lmonly.yaml
 create mode 100644 configs/stage2_ooc_finetune_lmonly.yaml


In [5]:
!python scripts/train_stage2.py --config configs/stage2_ooc_finetune_lmonly.yaml

Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/train_stage2.py", line 55, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/train_stage2.py", line 21, in main
    records = read_jsonl(Path(cfg.data_dir) / "cosmos_labeled_train.jsonl")
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/utils.py", line 164, in read_jsonl
    with open(path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/cosmos_labeled_train.jsonl'


In [6]:
!mkdir -p data/cosmos
!cp -r /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/* data/cosmos/
!python scripts/prepare_cosmos.py

^C
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 25, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 15, in main
    counts = prepare_all(
             ^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/data/cosmos.py", line 326, in prepare_all
    "train": prepare_unlabeled_split(
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/data/cosmos.py", line 129, in prepare_unlabeled_split
    raise FileNotFoundError(
FileNotFoundError: COSMOS train_data.json not found at data/cosmos/train_data.json (see docs/DATASETS.md).


In [7]:
!rm -rf data/cosmos
!mkdir -p data/cosmos
!ln -sf /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/* data/cosmos/

In [8]:
!ls data/cosmos/ | head


cosmos_anns
images_test_clean
images_train
images_val


In [9]:
!python scripts/prepare_cosmos.py

Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 25, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 15, in main
    counts = prepare_all(
             ^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/data/cosmos.py", line 326, in prepare_all
    "train": prepare_unlabeled_split(
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/data/cosmos.py", line 129, in prepare_unlabeled_split
    raise FileNotFoundError(
FileNotFoundError: COSMOS train_data.json not found at data/cosmos/train_data.json (see docs/DATASETS.md).


In [10]:
!ls data/cosmos/cosmos_anns/

test_data.json	train_data.json  val_data.json


In [11]:
!ln -sf data/cosmos/cosmos_anns/train_data.json data/cosmos/train_data.json
!ln -sf data/cosmos/cosmos_anns/val_data.json data/cosmos/val_data.json
!ln -sf data/cosmos/cosmos_anns/test_data.json data/cosmos/test_data.json

In [12]:
!cd data/cosmos && ln -sf cosmos_anns/train_data.json train_data.json && ln -sf cosmos_anns/val_data.json val_data.json && ln -sf cosmos_anns/test_data.json test_data.json

In [13]:
!ls -la data/cosmos/*.json

lrwxrwxrwx 1 root root 26 Sep 26 18:02 data/cosmos/test_data.json -> cosmos_anns/test_data.json
lrwxrwxrwx 1 root root 27 Sep 26 18:02 data/cosmos/train_data.json -> cosmos_anns/train_data.json
lrwxrwxrwx 1 root root 25 Sep 26 18:02 data/cosmos/val_data.json -> cosmos_anns/val_data.json


In [14]:
!python scripts/prepare_cosmos.py

[2026-09-26 18:13:23,534] INFO misinfodet.data.cosmos: COSMOS train: min_entity_freq=3 (spaCy PERSON) kept 83081/161754 images with a repeated-person caption
[2026-09-26 18:13:23,720] INFO misinfodet.data.cosmos: COSMOS train: wrote 12000 unlabeled records to data/cosmos_train.jsonl
[2026-09-26 18:15:49,849] INFO misinfodet.data.cosmos: COSMOS val: min_entity_freq=3 (spaCy PERSON) kept 18550/41006 images with a repeated-person caption
[2026-09-26 18:15:50,014] INFO misinfodet.data.cosmos: COSMOS val: wrote 12000 unlabeled records to data/cosmos_val.jsonl
[2026-09-26 18:15:50,249] INFO misinfodet.data.cosmos: COSMOS test: skipped 1700 records with missing image files (known artifact, see docstring)
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 25, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 15, in main
    counts = prepare_all(
             ^^^^^^^^^^^^
  F

In [15]:
!ls data/cosmos/images_test_clean/ | head -5

test


In [16]:
!grep -n "images_test_clean\|image_path\|img_local_path" src/misinfodet/data/cosmos.py | head -30

20:Schema: {"id", "image_path", "caption", "source"}
35:Schema: {"id", "image_path", "caption1", "caption2", "label", "source"}
71:def _image_path(cosmos_dir: str | Path, img_local_path: str) -> str:
72:    return str(Path(cosmos_dir) / img_local_path.lstrip("./"))
189:        image_path = _image_path(cosmos_dir, entry["img_local_path"])
193:            pair_key = (image_path, article["caption"])
200:                    "image_path": image_path,
242:        image_path = _image_path(cosmos_dir, entry["img_local_path"])
243:        if not Path(image_path).exists():
255:                f"No 'context_label' or 'label' field on entry {entry.get('img_local_path')}"
260:                "image_path": image_path,


In [17]:
!head -c 500 data/cosmos/test_data.json

{"img_local_path": "test/0.jpg", "caption1": "Julian Castro at his announcement in San Antonio, Tex., on Saturday. Mr. Castro, the former secretary of housing and urban development, would be one of the youngest presidents if elected.", "caption2": "Julian Castro at his announcement in San Antonio, Tex., on Saturday, Jan. 12, 2019.", "context_label": 0, "article_url": "https://www.nytimes.com/2019/06/13/us/politics/julian-castro-fox-town-hall.html", "maskrcnn_bboxes": [[389.9706726074219, 72.9228

In [18]:
!grep -n "cosmos_dir\|images_train\|images_val\|images_test" src/misinfodet/data/cosmos.py | grep -v "^[0-9]*:#" | head -40

4:images_train.zip, images_val.zip, images_test.zip (see docs/DATASETS.md):
71:def _image_path(cosmos_dir: str | Path, img_local_path: str) -> str:
72:    return str(Path(cosmos_dir) / img_local_path.lstrip("./"))
93:    cosmos_dir: str | Path,
127:    ann_path = Path(cosmos_dir) / f"{split}_data.json"
189:        image_path = _image_path(cosmos_dir, entry["img_local_path"])
219:def _load_test_records(cosmos_dir: str | Path, max_samples: int | None = None) -> list[dict]:
224:    artifact file present in the ORIGINAL COSMOS release's images_test.zip,
230:    ann_path = Path(cosmos_dir) / "test_data.json"
242:        image_path = _image_path(cosmos_dir, entry["img_local_path"])
276:    cosmos_dir: str | Path,
286:    records = _load_test_records(cosmos_dir, max_samples)
312:    cosmos_dir: str | Path,
327:            cosmos_dir, "train", out_dir / "cosmos_train.jsonl", max_samples,
331:            cosmos_dir, "val", out_dir / "cosmos_val.jsonl", max_samples,
336:    all_labeled = list(_l

In [19]:
!cd data/cosmos && ln -sf images_test_clean/test test


In [20]:
!ls data/cosmos/test/ | head -3

0.jpg
1000.png
1001.png
ls: write error: Broken pipe


In [21]:
!python scripts/prepare_cosmos.py

[2026-09-26 18:38:07,925] INFO misinfodet.data.cosmos: COSMOS train: min_entity_freq=3 (spaCy PERSON) kept 83081/161754 images with a repeated-person caption
[2026-09-26 18:38:08,112] INFO misinfodet.data.cosmos: COSMOS train: wrote 12000 unlabeled records to data/cosmos_train.jsonl
[2026-09-26 18:40:33,452] INFO misinfodet.data.cosmos: COSMOS val: min_entity_freq=3 (spaCy PERSON) kept 18550/41006 images with a repeated-person caption
[2026-09-26 18:40:33,605] INFO misinfodet.data.cosmos: COSMOS val: wrote 12000 unlabeled records to data/cosmos_val.jsonl
[2026-09-26 18:40:52,887] INFO misinfodet.data.cosmos: COSMOS test: skipped 1 records with missing image files (known artifact, see docstring)
[2026-09-26 18:40:53,724] INFO misinfodet.data.cosmos: COSMOS labeled split (seed=42): 1189 train / 254 val / 256 test
train: 12000 records
val: 12000 records
labeled_train: 1189 records
labeled_val: 254 records
labeled_test: 256 records


In [22]:
!mkdir -p outputs/checkpoints/stage1_best_local
!cp -r /kaggle/input/datasets/tebogoseopa20/stage1-best-v2/* outputs/checkpoints/stage1_best_local/

In [23]:
!git pull

Already up to date.


In [24]:
!python scripts/train_stage2.py --config configs/stage2_ooc_finetune_lmonly.yaml

Stage 2: 1189 OOC fine-tuning examples
Stage 2: 254 held-out validation examples
Stage 2: starting training with a fresh, randomly-initialized LoRA adapter
[2026-09-26 18:45:28,820] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 4.78MB/s]
model.safetensors.index.json: 70.1kB [00:00, 125MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [01:00<00:00, 20.08s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:11<00:00, 196MB/s]
Loading weights: 100%|█| 686/686 [00:07<00:00, 89.72it/s, Materializing param=mo
preprocessor_config.json: 100%|█████████████████| 505/505 [00:00<00:00, 296kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, 

In [1]:
!kaggle datasets files tebogoseopa20/stage2-checkpoints-lmonly-best

name                           size  creationDate                
-------------------------  --------  --------------------------  
README.md                      5187  2026-09-26 23:02:41.230000  
adapter_config.json            1028  2026-09-26 23:02:41.272000  
adapter_model.safetensors  76615400  2026-09-26 23:02:42.638000  
chat_template.jinja             674  2026-09-26 23:02:41.338000  
processor_config.json           754  2026-09-26 23:02:41.520000  
tokenizer.json              3619638  2026-09-26 23:02:41.484000  
tokenizer_config.json           557  2026-09-26 23:02:41.466000  


In [5]:
!mkdir -p outputs/checkpoints/stage2_best_lmonly_local
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-lmonly-best/* outputs/checkpoints/stage2_best_lmonly_local/

In [6]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_lmonly.yaml

Loading model...
[2026-09-27 06:07:54,293] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 3.65MB/s]
model.safetensors.index.json: 70.1kB [00:00, 111MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [01:02<00:00, 20.99s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:11<00:00, 198MB/s]
Loading weights: 100%|█| 686/686 [00:09<00:00, 73.37it/s, Materializing param=mo
preprocessor_config.json: 100%|█████████████████| 505/505 [00:00<00:00, 711kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
tokenizer_config.json: 1.45kB [00:00, 2.97MB/s]
tokenizer.json: 3.62MB [00:00, 69.5MB/s]
tok

In [7]:
!mkdir -p outputs/checkpoints/stage1_best_local outputs/checkpoints/stage2_best_lmonly_local
!cp -r /kaggle/input/datasets/tebogoseopa20/stage1-best-v2/* outputs/checkpoints/stage1_best_local/
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-lmonly-best/* outputs/checkpoints/stage2_best_lmonly_local/

In [8]:
!ls outputs/checkpoints/stage1_best_local/
!ls outputs/checkpoints/stage2_best_lmonly_local/

adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json
adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json


In [9]:
!rm -rf data/cosmos
!mkdir -p data/cosmos
!ln -sf /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/* data/cosmos/
!cd data/cosmos && ln -sf cosmos_anns/train_data.json train_data.json && ln -sf cosmos_anns/val_data.json val_data.json && ln -sf cosmos_anns/test_data.json test_data.json && ln -sf images_test_clean/test test

In [10]:
!python scripts/prepare_cosmos.py

[2026-09-27 06:25:22,070] INFO misinfodet.data.cosmos: COSMOS train: min_entity_freq=3 (spaCy PERSON) kept 83081/161754 images with a repeated-person caption
[2026-09-27 06:25:22,256] INFO misinfodet.data.cosmos: COSMOS train: wrote 12000 unlabeled records to data/cosmos_train.jsonl
[2026-09-27 06:27:49,008] INFO misinfodet.data.cosmos: COSMOS val: min_entity_freq=3 (spaCy PERSON) kept 18550/41006 images with a repeated-person caption
[2026-09-27 06:27:49,160] INFO misinfodet.data.cosmos: COSMOS val: wrote 12000 unlabeled records to data/cosmos_val.jsonl
[2026-09-27 06:27:57,565] INFO misinfodet.data.cosmos: COSMOS test: skipped 1 records with missing image files (known artifact, see docstring)
[2026-09-27 06:27:58,607] INFO misinfodet.data.cosmos: COSMOS labeled split (seed=42): 1189 train / 254 val / 256 test
train: 12000 records
val: 12000 records
labeled_train: 1189 records
labeled_val: 254 records
labeled_test: 256 records


In [11]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_lmonly.yaml

Loading model...
[2026-09-27 06:29:56,019] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 8278.23it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 146.42it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-27 06:30:06,727] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-27 06:30:07,356] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.

In [12]:
!ls scripts/

calibrate_threshold.py	    prepare_cosmos.py	      run_inference.py
compare_stage1_baseline.py  prepare_mmfakebench.py    train_stage1.py
diagnose_gap.py		    prepare_newsclippings.py  train_stage2.py
diagnose_stage1_data.py     run_ablations.sh
evaluate.py		    run_baseline.py


In [13]:
!python scripts/evaluate.py --help

usage: evaluate.py [-h] --predictions PREDICTIONS [--config CONFIG]
                   [--dataset-jsonl DATASET_JSONL]
                   [--export-human-eval EXPORT_HUMAN_EVAL]

options:
  -h, --help            show this help message and exit
  --predictions PREDICTIONS
  --config CONFIG
  --dataset-jsonl DATASET_JSONL
                        Source dataset JSONL (needed for human-eval export
                        captions)
  --export-human-eval EXPORT_HUMAN_EVAL
                        If >0, export this many items for human annotation


In [14]:
!python scripts/run_inference.py --help

usage: run_inference.py [-h] --config CONFIG [--split SPLIT]
                        [--dataset {cosmos,newsclippings,mmfakebench}]
                        [--resume]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --split SPLIT
  --dataset {cosmos,newsclippings,mmfakebench}
  --resume              skip ids already predicted


In [15]:
!grep -n "add_argument\|predictions_path\|out_path\|\.jsonl\"" scripts/run_inference.py | head -30

13:    ap.add_argument("--config", required=True)
14:    ap.add_argument("--split", default="test")
15:    ap.add_argument("--dataset", default="cosmos", choices=["cosmos", "newsclippings", "mmfakebench"])
16:    ap.add_argument("--resume", action="store_true", help="skip ids already predicted")
26:    filename = f"cosmos_labeled_{args.split}.jsonl" if args.dataset == "cosmos" else f"{args.dataset}_{args.split}.jsonl"
28:    out_path = Path(cfg.output_dir) / "predictions" / f"{cfg.run_name}_{args.dataset}_{args.split}.jsonl"
29:    out_path.parent.mkdir(parents=True, exist_ok=True)
32:    if args.resume and out_path.exists():
33:        done = {r["id"] for r in iter_jsonl(out_path)}
36:        out_path.unlink(missing_ok=True)
37:    write_run_meta(cfg, out_path.parent)
50:        append_jsonl(result, out_path)
54:    print("Predictions written to", out_path)


In [16]:
!python scripts/run_inference.py --config configs/stage2_eval_lmonly.yaml

[2026-09-27 06:55:44,367] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 16688.21it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 152.08it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-27 06:55:54,234] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-27 06:55:54,813] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarni

In [17]:
!python scripts/evaluate.py --predictions outputs/predictions/stage2_eval_lmonly_cosmos_test.jsonl --config configs/stage2_eval_lmonly.yaml

{
  "n": 256,
  "accuracy": 0.5039,
  "macro_f1": 0.355,
  "macro_precision": 0.551,
  "macro_recall": 0.5039,
  "per_class": {
    "consistent/real": {
      "precision": 0.6,
      "recall": 0.0234,
      "f1": 0.0451,
      "support": 128
    },
    "out-of-context/fake": {
      "precision": 0.502,
      "recall": 0.9844,
      "f1": 0.6649,
      "support": 128
    }
  },
  "confusion_matrix": [
    [
      3,
      125
    ],
    [
      2,
      126
    ]
  ],
  "parse_failure_rate": 0.0
}
Metrics written to outputs/predictions/stage2_eval_lmonly_cosmos_test_metrics.json


In [18]:
!grep -n "SERPER\|api_key\|X-API-KEY" src/misinfodet/retrieval/evidence_retriever.py

24:def _search_serper(query: str, api_key: str, k: int) -> list[dict]:
27:        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
39:def _search_bing(query: str, api_key: str, k: int) -> list[dict]:
42:        headers={"Ocp-Apim-Subscription-Key": api_key},
87:        self.api_key = os.environ.get(cfg.search_api_key_env, "")
90:        if self.provider == "offline" or (not self.api_key and corpus.exists()):
102:                    hits = _search_serper(q, self.api_key, per_query)
104:                    hits = _search_bing(q, self.api_key, per_query)


In [19]:
!grep -n "search_api_key_env" src/misinfodet/utils.py src/misinfodet/config.py 2>/dev/null

src/misinfodet/utils.py:78:    search_api_key_env: str = "SEARCH_API_KEY"


In [21]:
%env SEARCH_API_KEY=55c558096196c27f7f146fbdb5a9f6c5090d38dd

env: SEARCH_API_KEY=55c558096196c27f7f146fbdb5a9f6c5090d38dd


In [22]:
!python scripts/run_inference.py --config configs/stage2_eval_lmonly.yaml

[2026-09-27 08:31:54,926] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 50131.12it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 146.05it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-09-27 08:32:05,110] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-09-27 08:32:05,691] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarni

In [24]:
!python scripts/evaluate.py --predictions outputs/predictions/stage2_eval_lmonly_cosmos_test.jsonl --config configs/stage2_eval_lmonly.yaml

{
  "n": 256,
  "accuracy": 0.5156,
  "macro_f1": 0.3671,
  "macro_precision": 0.754,
  "macro_recall": 0.5156,
  "per_class": {
    "consistent/real": {
      "precision": 1.0,
      "recall": 0.0312,
      "f1": 0.0606,
      "support": 128
    },
    "out-of-context/fake": {
      "precision": 0.5079,
      "recall": 1.0,
      "f1": 0.6737,
      "support": 128
    }
  },
  "confusion_matrix": [
    [
      4,
      124
    ],
    [
      0,
      128
    ]
  ],
  "parse_failure_rate": 0.0
}
Metrics written to outputs/predictions/stage2_eval_lmonly_cosmos_test_metrics.json


In [25]:
!python scripts/prepare_mmfakebench.py --help

usage: prepare_mmfakebench.py [-h] [--config CONFIG]

options:
  -h, --help       show this help message and exit
  --config CONFIG


In [26]:
!kaggle datasets list -s mmfakebench

No datasets found


In [30]:
!ls configs/ | grep -i mmfake

In [31]:
!grep -n "mmfakebench\|dataset_dir\|data_root" src/misinfodet/utils.py | head -20

39:    mmfakebench_dir: str = "data/mmfakebench"


In [32]:
!cat scripts/prepare_mmfakebench.py

#!/usr/bin/env python
"""Prepare MMFakeBench test split into unified JSONL (evaluation only)."""
import argparse
from pathlib import Path

from misinfodet.data.mmfakebench import prepare
from misinfodet.utils import Config


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/base.yaml")
    args = ap.parse_args()
    cfg = Config.from_yaml(args.config)
    n = prepare(cfg.mmfakebench_dir, Path(cfg.data_dir) / "mmfakebench_test.jsonl")
    print(f"Prepared {n} MMFakeBench records")


if __name__ == "__main__":
    main()


In [33]:
!cat src/misinfodet/data/mmfakebench.py

"""MMFakeBench preparation (secondary benchmark, evaluation only).

Expected raw layout after download (see docs/DATASETS.md):

    data/mmfakebench/MMFakeBench_test/...      (images)
    data/mmfakebench/MMFakeBench_test.json     (annotations)

Each raw annotation has: image_path, text, gt_answers ("True"/"Fake"),
fake_cls (one of 12 sub-types, e.g. "original", "mismatch", "text_veracity...").

Unified schema mirrors NewsCLIPpings, with the fine-grained forgery type kept
for the mixed-source breakdown analysis in the final report:
    {"id", "image_path", "caption", "label", "forgery_type", "source"}
label: 0 = real, 1 = fake (any forgery source).
"""
from __future__ import annotations

import json
from pathlib import Path

from ..utils import get_logger, write_jsonl

log = get_logger(__name__)


def prepare(
    mmfakebench_dir: str | Path,
    out_path: str | Path,
    split: str = "test",
    max_samples: int | None = None,
) -> int:
    root = Path(mmfakebench_dir)
    ann_path = 

In [34]:
!cat docs/DATASETS.md

# Dataset acquisition

## COSMOS (primary, as of the dataset switch approved by Dr Olukanmi)

Access is gated behind a form: fill it out at
https://docs.google.com/forms/d/13kJQ2wlv7sxyXoaM1Ddon6Nq7dUJY_oftl-6xzwTGow —
after submitting, the confirmation page links directly to four files (no
separate approval wait):

- `cosmos_anns.zip` (~108MB) — annotations for all three splits
- `images_train.zip` (~18.6GB)
- `images_val.zip` (~4.4GB)
- `images_test.zip` (~312MB)

Extract so this exists:
```
data/cosmos/train_data.json
data/cosmos/val_data.json
data/cosmos/test_data.json
data/cosmos/train/...   (images)
data/cosmos/val/...     (images)
data/cosmos/test/...    (images)
```
Then run `python scripts/prepare_cosmos.py`.

**Known limitation, already discussed with and approved by Dr Olukanmi:**
only `test_data.json` (~1,700 images) carries an out-of-context label —
COSMOS's own method is self-supervised, so `train_data.json`/`val_data.json`
(160K+ images) have none. Stage 1 (news-domain e

In [ ]:
!pip install -q huggingface_hub

In [1]:
from huggingface_hub import HfApi
files = HfApi().list_repo_files("liuxuannan/MMFakeBench", repo_type="dataset", token="YOUR_HF_TOKEN")
print(len(files))
print(files[:30])

6
['.gitattributes', 'MMFakeBench_test.json', 'MMFakeBench_test.zip', 'MMFakeBench_val.json', 'MMFakeBench_val.zip', 'README.md']


In [2]:
from huggingface_hub import hf_hub_download
for f in ["MMFakeBench_test.json", "MMFakeBench_test.zip"]:
    hf_hub_download("liuxuannan/MMFakeBench", f, repo_type="dataset", local_dir="data/mmfakebench", token="YOUR_HF_TOKEN")

MMFakeBench_test.json: 0.00B [00:00, ?B/s]

MMFakeBench_test.zip:   0%|          | 0.00/6.81G [00:00<?, ?B/s]

In [3]:
!cd data/mmfakebench && unzip -q MMFakeBench_test.zip

In [4]:
!ls data/mmfakebench

MMFakeBench_test  MMFakeBench_test.json  MMFakeBench_test.zip


In [5]:
!python -c "import json; a=json.load(open('data/mmfakebench/MMFakeBench_test.json')); print(len(a)); print(a[0]['image_path'])"

10000
/real/bbc_test_500/bbc_test_0.png


In [6]:
!cd data/mmfakebench && for d in MMFakeBench_test/*; do ln -sfn "$d" "$(basename "$d")"; done

In [7]:
!ls -la data/mmfakebench

total 6652000
drwxr-xr-x 4 root root       4096 Oct  7 18:01 .
drwxr-xr-x 3 root root       4096 Oct  7 17:56 ..
drwxr-xr-x 3 root root       4096 Oct  7 17:55 .cache
lrwxrwxrwx 1 root root         21 Oct  7 18:01 fake -> MMFakeBench_test/fake
drwxrwxrwx 5 root root       4096 Mar 25  2025 MMFakeBench_test
-rw-r--r-- 1 root root    3138108 Oct  7 17:55 MMFakeBench_test.json
-rw-r--r-- 1 root root 6808483932 Oct  7 17:56 MMFakeBench_test.zip
lrwxrwxrwx 1 root root         21 Oct  7 18:01 real -> MMFakeBench_test/real
lrwxrwxrwx 1 root root         23 Oct  7 18:01 source -> MMFakeBench_test/source


In [12]:
!ls data/mmfakebench/real/bbc_test_500/bbc_test_0.png

data/mmfakebench/real/bbc_test_500/bbc_test_0.png


In [13]:
!python scripts/prepare_mmfakebench.py

python3: can't open file '/kaggle/working/scripts/prepare_mmfakebench.py': [Errno 2] No such file or directory


In [17]:
!ls /kaggle/working

data  Honours-Research-Project


In [18]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [19]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for misinfodet (pyproject.toml) ... done


In [20]:
!mkdir -p data && mv /kaggle/working/data/mmfakebench data/mmfakebench

In [21]:
!python scripts/prepare_mmfakebench.py

[2026-10-07 18:08:56,599] INFO misinfodet.data.mmfakebench: MMFakeBench test: wrote 10000 records to data/mmfakebench_test.jsonl
Prepared 10000 MMFakeBench records


In [22]:
!python scripts/calibrate_threshold.py --help

usage: calibrate_threshold.py [-h] --config CONFIG [--debug]

options:
  -h, --help       show this help message and exit
  --config CONFIG
  --debug          run tokenization sanity check on 1 example, then exit


In [23]:
!grep -n "cosmos\|mmfakebench\|dataset\|split\|max_samples" scripts/calibrate_threshold.py | head -30

31:def score_dataset(model, processor, records, cfg, label: str = ""):
106:        val_records = read_jsonl(Path(cfg.data_dir) / "cosmos_labeled_val.jsonl")
113:    val_records = read_jsonl(Path(cfg.data_dir) / "cosmos_labeled_val.jsonl")
114:    test_records = read_jsonl(Path(cfg.data_dir) / "cosmos_labeled_test.jsonl")
117:    val_scored = score_dataset(model, processor, val_records, cfg, label="val ")
124:    test_scored = score_dataset(model, processor, test_records, cfg, label="test ")


In [24]:
!python -c "import json,collections; r=[json.loads(l) for l in open('data/mmfakebench_test.jsonl')]; print(collections.Counter((x['label'],x['forgery_type']) for x in r))"

Counter({(0, 'original'): 3000, (1, 'textual_veracity_distortion'): 3000, (1, 'mismatch'): 3000, (1, 'visual_veracity_distortion'): 1000})


In [25]:
!cat scripts/calibrate_threshold.py

#!/usr/bin/env python
"""Calibrate the out-of-context/consistent decision threshold using the
model's actual token-level confidence, rather than trusting whichever word
it happens to generate first in free text.

Standard technique: force the model to the point right after "VERDICT:"
and read its probability for the next token starting "consistent" vs
"out-of-context" — a continuous score in [0,1] instead of a hard binary
guess. That score gets thresholded using the validation set (never
trained on), which can correct the class bias seen across every
configuration tested so far without any further training.

IMPORTANT: run with --debug first, on one example, to confirm the
tokenization assumption this relies on actually holds for this model,
before trusting a full calibration run.
"""
import argparse
from pathlib import Path

from PIL import Image

from misinfodet.models.model_factory import (
    debug_verdict_tokenization,
    get_verdict_score,
    load_llava,
)
from misinfodet.pipe

In [26]:
!cat src/misinfodet/pipeline/prompts.py

"""All prompt templates for the system, centralised for auditability.

Keeping every prompt here makes the methodology chapter easy to write and the
system easy to ablate: any prompt change is a single, version-controlled diff.
"""

# --------------------------------------------------------------------------- #
# Stage 1: news-domain entity alignment
# --------------------------------------------------------------------------- #
STAGE1_ALIGNMENT_PROMPTS = [
    "Describe this news photograph, naming the specific people, places, and events shown.",
    "Write a news caption for this image, identifying the named entities involved.",
    "What is happening in this news image? Identify who is pictured and where.",
]

# --------------------------------------------------------------------------- #
# Stage 2 / inference: internal consistency verdict
# --------------------------------------------------------------------------- #
VERDICT_PROMPT = (
    "You are a fact-checking assistant analysi

In [27]:
!grep -n "def get_verdict_score" -A 30 src/misinfodet/models/model_factory.py

203:def get_verdict_score(model, processor, image, prompt: str, cfg: Config) -> float:
204-    """Returns P(out-of-context) as a continuous score in [0, 1], via one
205-    forward pass rather than full text generation. Sums probability across
206-    every "out"-starting vs "cons"-starting token among the top candidates,
207-    rather than comparing two fixed token ids — confirmed necessary since
208-    the model's real outputs vary in capitalization (Out/out/OUT), and a
209-    naive single-token comparison shares a tokenizer artifact token between
210-    the two classes (see debug_verdict_tokenization).
211-    """
212-    import torch
213-
214-    conversation = [
215-        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]},
216-    ]
217-    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
218-    forced_text = prompt_text + "VERDICT:"
219-
220-    inputs = processor(images=image, text=forced_text, return_te

In [28]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.21 KiB | 1.10 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   a447483..29aea69  main       -> origin/main
Updating a447483..29aea69
Fast-forward
 scripts/calibrate_mmfakebench.py | 121 +++++++++++++++++++++++++++++++++++++++
 1 file changed, 121 insertions(+)
 create mode 100644 scripts/calibrate_mmfakebench.py


In [29]:
!ls /kaggle/input/datasets/tebogoseopa20/

cosmos-dataset	stage2-best			stage2-checkpoints-resumed-best
mmfakebench	stage2-best-3ep-manual
stage1-best-v2	stage2-checkpoints-lmonly-best


In [30]:
!ls configs/ | grep lmonly

stage2_eval_lmonly.yaml
stage2_ooc_finetune_lmonly.yaml


In [31]:
!find /kaggle/input/datasets/tebogoseopa20/stage1-best-v2 /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-lmonly-best -name adapter_config.json

/kaggle/input/datasets/tebogoseopa20/stage1-best-v2/adapter_config.json
/kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-lmonly-best/adapter_config.json


In [36]:
!mkdir -p outputs/checkpoints

In [37]:
!cp -r /kaggle/input/datasets/tebogoseopa20/stage1-best-v2 outputs/checkpoints/stage1_best_local

In [38]:
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-lmonly-best outputs/checkpoints/stage2_best_lmonly_local

In [39]:
!git pull

Already up to date.


In [40]:
!ls scripts/calibrate_mmfakebench.py

scripts/calibrate_mmfakebench.py


In [41]:
!python scripts/calibrate_mmfakebench.py --config configs/stage2_eval_lmonly.yaml

Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-07 18:26:17,889] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 3.76MB/s]
model.safetensors.index.json: 70.1kB [00:00, 143MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:58<00:00, 19.41s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:09<00:00, 203MB/s]
Loading weights: 100%|█| 686/686 [00:07<00:00, 97.64it/s, Materializing param=mo
preprocessor_config.json: 100%|█████████████████| 505/505 [00:00<00:00, 936kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
tokenizer_config.json: 1.45kB [00:00, 4.14MB/s]
t

In [42]:
!python scripts/calibrate_mmfakebench.py --config configs/stage2_eval_lmonly.yaml --n 10000

Sample: 5000 calibration + 5000 evaluation
Loading model...
[2026-10-07 19:03:43,622] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 17427.86it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:05<00:00, 129.42it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-07 19:03:57,993] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-10-07 19:03:58,570] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3

In [43]:
!git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.27 KiB | 260.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   29aea69..e74a040  main       -> origin/main
Updating 29aea69..e74a040
Fast-forward
 configs/base_llava_eval.yaml     |  8 ++++++++
 scripts/calibrate_mmfakebench.py | 13 ++++++++-----
 scripts/calibrate_threshold.py   | 24 ++++++++++++++++++++----
 3 files changed, 36 insertions(+), 9 deletions(-)
 create mode 100644 configs/base_llava_eval.yaml


In [44]:
!python scripts/calibrate_mmfakebench.py --config configs/stage2_eval_lmonly.yaml --n 2000 --blank-image

Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-07 21:09:09,889] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 29127.11it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 159.90it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-07 21:09:23,112] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-10-07 21:09:23,668] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3

In [71]:
!mkdir -p data/cosmos

In [72]:
!ln -sf /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/* data/cosmos/

In [74]:
!cd data/cosmos && ln -sf cosmos_anns/train_data.json train_data.json && ln -sf cosmos_anns/val_data.json val_data.json && ln -sf cosmos_anns/test_data.json test_data.json

In [76]:
!ls -la data/cosmos/ | grep test

lrwxrwxrwx 1 root root   69 Oct  7 21:46 images_test_clean -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_test_clean
lrwxrwxrwx 1 root root   22 Oct  7 21:40 test -> images_test_clean/test
lrwxrwxrwx 1 root root   26 Oct  7 21:46 test_data.json -> cosmos_anns/test_data.json


In [77]:
!python scripts/prepare_cosmos.py

[2026-10-07 21:57:42,164] INFO misinfodet.data.cosmos: COSMOS train: min_entity_freq=3 (spaCy PERSON) kept 83081/161754 images with a repeated-person caption
[2026-10-07 21:57:42,354] INFO misinfodet.data.cosmos: COSMOS train: wrote 12000 unlabeled records to data/cosmos_train.jsonl
[2026-10-07 22:00:14,386] INFO misinfodet.data.cosmos: COSMOS val: min_entity_freq=3 (spaCy PERSON) kept 18550/41006 images with a repeated-person caption
[2026-10-07 22:00:14,553] INFO misinfodet.data.cosmos: COSMOS val: wrote 12000 unlabeled records to data/cosmos_val.jsonl
[2026-10-07 22:00:18,886] INFO misinfodet.data.cosmos: COSMOS test: skipped 1 records with missing image files (known artifact, see docstring)
[2026-10-07 22:00:19,772] INFO misinfodet.data.cosmos: COSMOS labeled split (seed=42): 1189 train / 254 val / 256 test
train: 12000 records
val: 12000 records
labeled_train: 1189 records
labeled_val: 254 records
labeled_test: 256 records


In [78]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_lmonly.yaml

Loading model...
[2026-10-07 22:03:38,927] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 19784.45it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 165.36it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-07 22:03:51,980] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-10-07 22:03:52,567] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.

In [79]:
!python scripts/calibrate_threshold.py --config configs/stage2_eval_lmonly.yaml --blank-image

Loading model...
[2026-10-07 22:14:30,185] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 43539.49it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 162.60it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-07 22:14:43,379] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
[2026-10-07 22:14:43,948] INFO misinfodet.models.model_factory: Applying stage2 adapter from outputs/checkpoints/stage2_best_lmonly_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.

In [80]:
!python scripts/calibrate_threshold.py --config configs/base_llava_eval.yaml

Loading model...
[2026-10-07 22:29:16,820] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 36472.21it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 159.70it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/calibrate_threshold.py", line 146, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/calibrate_threshold.py", line 115, in main
    model, processor = load_llava(cfg, trainable=False)
                       ^^^^^

In [85]:
!sed -i 's/^\(\s*\)model\.merge_adapter()/\1if hasattr(model, "merge_adapter"):\n\1    model.merge_adapter()/' src/misinfodet/models/model_factory.py

In [86]:
!grep -n -B1 -A1 "merge_adapter()" src/misinfodet/models/model_factory.py

70-    if not trainable:
71:        # merge_adapter() merges every nested adapter's LoRA deltas into its
72-        # base layer IN PLACE, without unloading/removing the PeftModel
--
75-        # inner model's `peft_config`, which is a property with no deleter.
76:        # merge_adapter() walks every submodule regardless of nesting depth,
77-        # so one call merges both stage1 and stage2 in a single pass.
--
82-                        if hasattr(model, "merge_adapter"):
83:                            model.merge_adapter()
84-


In [87]:
!sed -n '66,86p' src/misinfodet/models/model_factory.py

        if adapter:
            log.info("Applying %s adapter from %s", name, adapter)
            model = PeftModel.from_pretrained(model, adapter)

    if not trainable:
        # merge_adapter() merges every nested adapter's LoRA deltas into its
        # base layer IN PLACE, without unloading/removing the PeftModel
        # wrapper. merge_and_unload() (unload + remove) was tried first but
        # crashes on a doubly-nested PeftModel: it ends by deleting the
        # inner model's `peft_config`, which is a property with no deleter.
        # merge_adapter() walks every submodule regardless of nesting depth,
        # so one call merges both stage1 and stage2 in a single pass.
        if hasattr(model, "merge_adapter"):
            if hasattr(model, "merge_adapter"):
                if hasattr(model, "merge_adapter"):
                    if hasattr(model, "merge_adapter"):
                        if hasattr(model, "merge_adapter"):
                            model.merge_adapter(

In [92]:
!sed -i -e '79,82d' -e '83s/^ */            /' src/misinfodet/models/model_factory.py

In [93]:
!sed -n '70,82p' src/misinfodet/models/model_factory.py

    if not trainable:
        # merge_adapter() merges every nested adapter's LoRA deltas into its
        # base layer IN PLACE, without unloading/removing the PeftModel
        # wrapper. merge_and_unload() (unload + remove) was tried first but
        # crashes on a doubly-nested PeftModel: it ends by deleting the
        # inner model's `peft_config`, which is a property with no deleter.
        # merge_adapter() walks every submodule regardless of nesting depth,
        # so one call merges both stage1 and stage2 in a single pass.
        if hasattr(model, "merge_adapter"):
            # cfg.lora_r etc. here, and mismatches against the ORIGINAL
            # values used to train it aren't possible via this path.
            log.info("Resuming training from existing adapter at %s", resume_from)
            model = PeftModel.from_pretrained(model, resume_from, is_trainable=True)


In [95]:
!git checkout -- src/misinfodet/models/model_factory.py

In [96]:
p = "src/misinfodet/models/model_factory.py"
s = open(p).read()
new = '        if hasattr(model, "merge_adapter"):\n            model.merge_adapter()\n'
if new not in s:
    s = s.replace("        model.merge_adapter()\n", new, 1)
    open(p, "w").write(s)
print(s[s.index("if not trainable:"):s.index("if trainable:")])

if not trainable:
        # merge_adapter() merges every nested adapter's LoRA deltas into its
        # base layer IN PLACE, without unloading/removing the PeftModel
        # wrapper. merge_and_unload() (unload + remove) was tried first but
        # crashes on a doubly-nested PeftModel: it ends by deleting the
        # inner model's `peft_config`, which is a property with no deleter.
        # merge_adapter() walks every submodule regardless of nesting depth,
        # so one call merges both stage1 and stage2 in a single pass.
        if hasattr(model, "merge_adapter"):
            model.merge_adapter()

    


In [97]:
!python scripts/calibrate_threshold.py --config configs/base_llava_eval.yaml
!python scripts/calibrate_mmfakebench.py --config configs/base_llava_eval.yaml --n 2000

Loading model...
[2026-10-07 22:37:49,713] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 27962.03it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 165.12it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Scoring 254 validation examples...
  val 20/254 done
  val 40/254 done
  val 60/254 done
  val 80/254 done
  val 100/254 done
  val 120/254 done
  val 140/254 done
  val 160/254 done
  val 180/254 done
  val 200/254 done
  val 220/254 done
  val 240/254 done

--- Validation @ default 0.5 threshold (threshold=0.50) ---
n=254  accurac

In [98]:
!python scripts/calibrate_threshold.py --config configs/diag_stage1_only.yaml

Loading model...
[2026-10-07 23:23:09,709] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 5899.16it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 159.80it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-07 23:23:22,827] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Scoring 254 vali

In [99]:
!grep -n "def \|class " src/misinfodet/retrieval/evidence_retriever.py

24:def _search_serper(query: str, api_key: str, k: int) -> list[dict]:
39:def _search_bing(query: str, api_key: str, k: int) -> list[dict]:
54:class OfflineCorpus:
61:    def __init__(self, corpus_path):
73:    def search(self, query: str, k: int) -> list[dict]:
83:class EvidenceRetriever:
84:    def __init__(self, cfg: Config):
96:    def retrieve(self, queries: list[str]) -> list[dict]:


In [100]:
!grep -n "def \|class " src/misinfodet/pipeline/*.py | grep -v prompts

src/misinfodet/pipeline/detector.py:38:class DetectionResult:
src/misinfodet/pipeline/detector.py:50:    def to_dict(self) -> dict:
src/misinfodet/pipeline/detector.py:65:def _extract_field(text: str, name: str) -> str:
src/misinfodet/pipeline/detector.py:70:class Detector:
src/misinfodet/pipeline/detector.py:71:    def __init__(self, model, processor, cfg: Config):
src/misinfodet/pipeline/detector.py:77:    def detect(self, record: dict) -> DetectionResult:
src/misinfodet/pipeline/detector.py:136:            # class 0 and is flagged, so parse failures are visible in analysis


In [101]:
!grep -n "generate\|max_new_tokens" src/misinfodet/models/model_factory.py

113:def generate(model, processor, image, prompt: str, cfg: Config) -> str:
126:        output_ids = model.generate(
128:            max_new_tokens=cfg.max_new_tokens,
132:    generated = output_ids[0][inputs["input_ids"].shape[1]:]
133:    return processor.decode(generated, skip_special_tokens=True).strip()
136:def generate_text_only(model, processor, prompt: str, cfg: Config) -> str:
144:        output_ids = model.generate(
145:            **inputs, max_new_tokens=128, do_sample=False
147:    generated = output_ids[0][inputs["input_ids"].shape[1]:]
148:    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()


In [102]:
!python scripts/calibrate_mmfakebench.py --config configs/diag_stage1_only.yaml --n 2000

Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-07 23:40:39,883] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 33112.93it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 163.26it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-07 23:40:52,940] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to roundin

In [103]:
!git pull

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.01 KiB | 2.01 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   e74a040..dcf1a9c  main       -> origin/main
Updating e74a040..dcf1a9c
Fast-forward
 scripts/calibrate_threshold.py |  9 +++++
 scripts/compare_runs.py        | 91 ++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 100 insertions(+)
 create mode 100644 scripts/compare_runs.py


In [104]:
!python scripts/calibrate_mmfakebench.py --config configs/diag_stage1_only.yaml --n 2000 --cosmos-threshold 0.81

Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-08 00:14:06,854] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 17822.82it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 160.85it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-08 00:14:19,980] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to roundin

In [105]:
!ls outputs/predictions/ | grep scores

base_llava_eval_mmfakebench_scores.jsonl
diag_stage1_only_mmfakebench_scores.jsonl
stage2_eval_lmonly_mmfakebench_blank_scores.jsonl
stage2_eval_lmonly_mmfakebench_scores.jsonl


In [106]:
!python scripts/compare_runs.py --mmfb outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl outputs/predictions/base_llava_eval_mmfakebench_scores.jsonl --n 2000

A = outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl  (threshold 0.74)
B = outputs/predictions/base_llava_eval_mmfakebench_scores.jsonl  (threshold 0.80)
n evaluation items = 1000, bootstrap iterations = 5000

AUROC     A-B = +0.0280   95% CI [+0.0151, +0.0413]   p~0.000   (significant)
Accuracy  A-B = +0.0220   95% CI [+0.0030, +0.0420]   p~0.030   (significant)


In [107]:
!python scripts/compare_runs.py --mmfb outputs/predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl --n 2000
!python scripts/compare_runs.py --mmfb outputs/predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl outputs/predictions/stage2_eval_lmonly_mmfakebench_blank_scores.jsonl --n 2000

A = outputs/predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl  (threshold 0.81)
B = outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl  (threshold 0.74)
n evaluation items = 1000, bootstrap iterations = 5000

AUROC     A-B = -0.0057   95% CI [-0.0181, +0.0067]   p~0.387   (not significant)
Accuracy  A-B = -0.0170   95% CI [-0.0360, +0.0020]   p~0.078   (not significant)
A = outputs/predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl  (threshold 0.81)
B = outputs/predictions/stage2_eval_lmonly_mmfakebench_blank_scores.jsonl  (threshold 0.88)
n evaluation items = 1000, bootstrap iterations = 5000

AUROC     A-B = +0.0138   95% CI [-0.0187, +0.0460]   p~0.406   (not significant)
Accuracy  A-B = -0.0070   95% CI [-0.0380, +0.0240]   p~0.680   (not significant)


In [108]:
!python scripts/calibrate_threshold.py --config configs/base_llava_eval.yaml
!python scripts/calibrate_threshold.py --config configs/diag_stage1_only.yaml
!python scripts/calibrate_threshold.py --config configs/stage2_eval_lmonly.yaml
!python scripts/calibrate_threshold.py --config configs/stage2_eval_lmonly.yaml --blank-image

Loading model...
[2026-10-08 00:33:06,170] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 25420.02it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 162.51it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Scoring 254 validation examples...
  val 20/254 done
  val 40/254 done
  val 60/254 done
  val 80/254 done
  val 100/254 done
  val 120/254 done
  val 140/254 done
  val 160/254 done
  val 180/254 done
  val 200/254 done
  val 220/254 done
  val 240/254 done

--- Validation @ default 0.5 threshold (threshold=0.50) ---
n=254  accurac

In [109]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_cosmos_scores.jsonl outputs/predictions/base_llava_eval_cosmos_scores.jsonl
!python scripts/compare_runs.py --cosmos outputs/predictions/stage2_eval_lmonly_cosmos_scores.jsonl outputs/predictions/diag_stage1_only_cosmos_scores.jsonl
!python scripts/compare_runs.py --cosmos outputs/predictions/stage2_eval_lmonly_cosmos_scores.jsonl outputs/predictions/stage2_eval_lmonly_blank_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
B = outputs/predictions/base_llava_eval_cosmos_scores.jsonl  (threshold 0.85)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.0354   95% CI [+0.0195, +0.0529]   p~0.000   (significant)
Accuracy  A-B = +0.0469   95% CI [+0.0117, +0.0859]   p~0.018   (significant)
A = outputs/predictions/stage2_eval_lmonly_cosmos_scores.jsonl  (threshold 0.85)
B = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = -0.0169   95% CI [-0.0315, -0.0033]   p~0.013   (significant)
Accuracy  A-B = -0.0195   95% CI [-0.0586, +0.0156]   p~0.340   (not significant)
A = outputs/predictions/stage2_eval_lmonly_cosmos_scores.jsonl  (threshold 0.85)
B = outputs/predictions/stage2_eval_lmonly_blank_cosmos_scores.jsonl  (threshold 0.91)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.0575   95% CI [

In [110]:
!sed -n '70,140p' src/misinfodet/pipeline/detector.py

class Detector:
    def __init__(self, model, processor, cfg: Config):
        self.model = model
        self.processor = processor
        self.cfg = cfg
        self.retriever = EvidenceRetriever(cfg) if cfg.use_retrieval else None

    def detect(self, record: dict) -> DetectionResult:
        image = Image.open(record["image_path"]).convert("RGB")
        paired = "caption1" in record and "caption2" in record
        if paired:
            caption1, caption2 = record["caption1"], record["caption2"]
            # Used only for retrieval query generation, which needs one text
            # blob — not a task-format assumption, see query_generator.py.
            caption_for_retrieval = f'Caption 1: "{caption1}" | Caption 2: "{caption2}"'
            initial_prompt = PAIRED_VERDICT_PROMPT.format(caption1=caption1, caption2=caption2)
        else:
            caption = record["caption"]
            caption_for_retrieval = caption
            initial_prompt = VERDICT_PROMPT.format(capti

In [111]:
!cd /kaggle/working/Honours-Research-Project && zip -r /kaggle/working/predictions_backup.zip outputs/predictions

  adding: outputs/predictions/ (stored 0%)
  adding: outputs/predictions/stage2_eval_lmonly_blank_cosmos_scores.jsonl (deflated 82%)
  adding: outputs/predictions/diag_stage1_only_cosmos_scores.jsonl (deflated 82%)
  adding: outputs/predictions/stage2_eval_lmonly_mmfakebench_blank_scores.jsonl (deflated 76%)
  adding: outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl (deflated 76%)
  adding: outputs/predictions/stage2_eval_lmonly_cosmos_scores.jsonl (deflated 82%)
  adding: outputs/predictions/base_llava_eval_cosmos_scores.jsonl (deflated 82%)
  adding: outputs/predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl (deflated 76%)
  adding: outputs/predictions/base_llava_eval_mmfakebench_scores.jsonl (deflated 76%)


In [112]:
%cd /kaggle/working

/kaggle/working


In [113]:
from IPython.display import FileLink
FileLink("predictions_backup.zip")

/kaggle/working/predictions_backup.zip

In [114]:
!mkdir -p /kaggle/working/pred_ds && cp /kaggle/working/Honours-Research-Project/outputs/predictions/* /kaggle/working/pred_ds/

In [115]:
import json
json.dump({"title": "misinfodet-predictions", "id": "tebogoseopa20/misinfodet-predictions", "licenses": [{"name": "CC0-1.0"}]}, open("/kaggle/working/pred_ds/dataset-metadata.json", "w"))

In [116]:
!kaggle datasets create -p /kaggle/working/pred_ds

Starting upload for file stage2_eval_lmonly_blank_cosmos_scores.jsonl
100%|██████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 49.6kB/s]
Upload successful: stage2_eval_lmonly_blank_cosmos_scores.jsonl (38KB)
Starting upload for file diag_stage1_only_cosmos_scores.jsonl
100%|██████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 48.0kB/s]
Upload successful: diag_stage1_only_cosmos_scores.jsonl (38KB)
Starting upload for file stage2_eval_lmonly_mmfakebench_blank_scores.jsonl
100%|█████████████████████████████████████████| 105k/105k [00:00<00:00, 108kB/s]
Upload successful: stage2_eval_lmonly_mmfakebench_blank_scores.jsonl (105KB)
Starting upload for file diag_stage1_only_mmfakebench_scores.jsonl
100%|█████████████████████████████████████████| 105k/105k [00:00<00:00, 112kB/s]
Upload successful: diag_stage1_only_mmfakebench_scores.jsonl (105KB)
Starting upload for file stage2_eval_lmonly_cosmos_scores.jsonl
100%|██████████████████████████████████████| 37.6k/37.6k [

In [117]:
!kaggle datasets files tebogoseopa20/misinfodet-predictions

name                                                 size  creationDate                
-------------------------------------------------  ------  --------------------------  
base_llava_eval_cosmos_scores.jsonl                 38476  2026-10-08 01:28:04.790000  
base_llava_eval_mmfakebench_scores.jsonl           107522  2026-10-08 01:28:04.957000  
diag_stage1_only_cosmos_scores.jsonl                38479  2026-10-08 01:28:04.859000  
diag_stage1_only_mmfakebench_scores.jsonl          107550  2026-10-08 01:28:04.854000  
stage2_eval_lmonly_blank_cosmos_scores.jsonl        38495  2026-10-08 01:28:04.857000  
stage2_eval_lmonly_cosmos_scores.jsonl              38499  2026-10-08 01:28:04.863000  
stage2_eval_lmonly_mmfakebench_blank_scores.jsonl  107550  2026-10-08 01:28:04.929000  
stage2_eval_lmonly_mmfakebench_scores.jsonl        537684  2026-10-08 01:28:04.835000  


In [118]:
!cd /kaggle/working/Honours-Research-Project && git status --short

 M src/misinfodet/models/model_factory.py
?? data/
?? outputs/checkpoints/
?? outputs/predictions/
?? scripts/__pycache__/
?? src/misinfodet.egg-info/
?? src/misinfodet/__pycache__/__init__.cpython-312.pyc
?? src/misinfodet/__pycache__/utils.cpython-312.pyc
?? src/misinfodet/data/__pycache__/__init__.cpython-312.pyc
?? src/misinfodet/data/__pycache__/cosmos.cpython-312.pyc
?? src/misinfodet/data/__pycache__/instruction_builder.cpython-312.pyc
?? src/misinfodet/data/__pycache__/mmfakebench.cpython-312.pyc
?? src/misinfodet/evaluation/__pycache__/
?? src/misinfodet/models/__pycache__/
?? src/misinfodet/pipeline/__pycache__/
?? src/misinfodet/retrieval/__pycache__/
?? tests/__pycache__/


In [3]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [4]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for misinfodet (pyproject.toml) ... done


In [15]:
!mkdir -p data/cosmos

In [26]:
!ln -sf /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/* data/cosmos/

In [27]:
!cd data/cosmos && ln -sf cosmos_anns/train_data.json train_data.json && ln -sf cosmos_anns/val_data.json val_data.json && ln -sf cosmos_anns/test_data.json test_data.json

In [35]:
!cd data/cosmos && ln -sfn images_test_clean/test test

In [36]:
!python scripts/prepare_cosmos.py

[2026-10-08 10:37:07,871] INFO misinfodet.data.cosmos: COSMOS train: min_entity_freq=3 (spaCy PERSON) kept 83081/161754 images with a repeated-person caption
[2026-10-08 10:37:08,060] INFO misinfodet.data.cosmos: COSMOS train: wrote 12000 unlabeled records to data/cosmos_train.jsonl
[2026-10-08 10:39:42,024] INFO misinfodet.data.cosmos: COSMOS val: min_entity_freq=3 (spaCy PERSON) kept 18550/41006 images with a repeated-person caption
[2026-10-08 10:39:42,179] INFO misinfodet.data.cosmos: COSMOS val: wrote 12000 unlabeled records to data/cosmos_val.jsonl
[2026-10-08 10:39:49,363] INFO misinfodet.data.cosmos: COSMOS test: skipped 1 records with missing image files (known artifact, see docstring)
[2026-10-08 10:39:50,188] INFO misinfodet.data.cosmos: COSMOS labeled split (seed=42): 1189 train / 254 val / 256 test
train: 12000 records
val: 12000 records
labeled_train: 1189 records
labeled_val: 254 records
labeled_test: 256 records


In [38]:
!python scripts/check_leakage.py

python3: can't open file '/kaggle/working/Honours-Research-Project/scripts/check_leakage.py': [Errno 2] No such file or directory


In [39]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.73 KiB | 1.36 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   dcf1a9c..97546ba  main       -> origin/main
Updating dcf1a9c..97546ba
Fast-forward
 scripts/check_leakage.py | 157 +++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 157 insertions(+)
 create mode 100644 scripts/check_leakage.py


In [40]:
!ls scripts/check_leakage.py

scripts/check_leakage.py


In [41]:
!python scripts/check_leakage.py

  eval val: 0 unreadable images
  eval test: 0 unreadable images

=== Overlap of labeled val/test items with each reference set ===
  stage1_train: 2000/12000 images hashed
  stage1_train: 4000/12000 images hashed
  stage1_train: 6000/12000 images hashed
  stage1_train: 8000/12000 images hashed
  stage1_train: 10000/12000 images hashed
  stage1_train: 12000/12000 images hashed
  stage1_train: 12000 unreadable images
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/check_leakage.py", line 157, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/check_leakage.py", line 122, in main
    img_near = (hamming_min(eval_dh[s], r_dh) <= args.max_hamming) & eval_ok[s]
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/scripts/check_leakage.py", line 63, in hamming_min
    return np.unpackbits(x, axis=2).sum(axis=2).min(axis=1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [42]:
!head -c 400 data/cosmos_train.jsonl

{"id": "cosmos_train_0_0", "image_path": "data/cosmos/train/86724.jpg", "caption": "Roughneck Brian Waldner prepares pipe on a True Company oil drilling rig outside Watford, North Dakota, October 20, 2012.", "source": "cosmos"}
{"id": "cosmos_train_1_0", "image_path": "data/cosmos/train/65272.jpg", "caption": "North Korean leader Kim Jong Un and his wife Ri Sol Ju watch South Korean K-pop singers 

In [43]:
!ls data/cosmos/images_train | head -3

train


In [44]:
!ls data/cosmos/images_val | head -3

val


In [50]:
!cd data/cosmos && ln -sfn images_train/train train && ln -sfn images_val/val val

In [51]:
!ls data/cosmos/train/86724.jpg

data/cosmos/train/86724.jpg


In [52]:
!python scripts/check_leakage.py

  eval val: 0 unreadable images
  eval test: 0 unreadable images

=== Overlap of labeled val/test items with each reference set ===
  stage1_train: 2000/12000 images hashed
  stage1_train: 4000/12000 images hashed
  stage1_train: 6000/12000 images hashed
  stage1_train: 8000/12000 images hashed
  stage1_train: 10000/12000 images hashed
  stage1_train: 12000/12000 images hashed
  stage1_train: 0 unreadable images
val  vs stage1_train (n_ref=12000): img exact   1 | img near   2 | cap exact   1 | cap near   1 | ANY   2/254 (real 1, ooc 1)
test vs stage1_train (n_ref=12000): img exact   0 | img near   0 | cap exact   0 | cap near   0 | ANY   0/256 (real 0, ooc 0)
  stage1_val: 2000/12000 images hashed
  stage1_val: 4000/12000 images hashed
  stage1_val: 6000/12000 images hashed
  stage1_val: 8000/12000 images hashed
  stage1_val: 10000/12000 images hashed
  stage1_val: 12000/12000 images hashed
  stage1_val: 0 unreadable images
val  vs stage1_val   (n_ref=12000): img exact   0 | img near  

In [53]:
!mkdir -p outputs/predictions && cp /kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/*.jsonl outputs/predictions/

In [55]:
import json
from sklearn.metrics import roc_auc_score

def macro_f1(preds, golds):
    f = []
    for c in (0, 1):
        tp = sum(p == c and g == c for p, g in zip(preds, golds))
        fp = sum(p == c and g != c for p, g in zip(preds, golds))
        fn = sum(p != c and g == c for p, g in zip(preds, golds))
        pr = tp / (tp + fp) if tp + fp else 0
        rc = tp / (tp + fn) if tp + fn else 0
        f.append(2 * pr * rc / (pr + rc) if pr + rc else 0)
    return sum(f) / 2

def best_t(scored):
    golds = [g for _, g in scored]
    return max((i / 100 for i in range(5, 96)), key=lambda t: macro_f1([int(s >= t) for s, _ in scored], golds))

flags = json.load(open("outputs/predictions/leakage_flags.json"))
bad = set(flags["val"] + flags["test"])
for run in ["base_llava_eval", "diag_stage1_only", "stage2_eval_lmonly"]:
    rows = [json.loads(l) for l in open(f"outputs/predictions/{run}_cosmos_scores.jsonl")]
    for name, keep in (("ALL", lambda r: True), ("CLEAN", lambda r: r["id"] not in bad)):
        v = [(r["score"], r["label"]) for r in rows if r["split"] == "val" and keep(r)]
        t = [(r["score"], r["label"]) for r in rows if r["split"] == "test" and keep(r)]
        th = best_t(v)
        auc = roc_auc_score([g for _, g in t], [s for s, _ in t])
        acc = sum((s >= th) == g for s, g in t) / len(t)
        print(f"{run:20s} {name:5s} n={len(t)} AUROC {auc:.4f} acc {acc:.4f} t={th:.2f}")

base_llava_eval      ALL   n=256 AUROC 0.8745 acc 0.7852 t=0.85
base_llava_eval      CLEAN n=130 AUROC 0.8610 acc 0.8308 t=0.87
diag_stage1_only     ALL   n=256 AUROC 0.9099 acc 0.8320 t=0.81
diag_stage1_only     CLEAN n=130 AUROC 0.9070 acc 0.8385 t=0.82
stage2_eval_lmonly   ALL   n=256 AUROC 0.8929 acc 0.8125 t=0.85
stage2_eval_lmonly   CLEAN n=130 AUROC 0.8870 acc 0.8385 t=0.88


In [56]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [57]:
!python scripts/calibrate_threshold.py --config configs/diag_stage1_only.yaml --blank-image

Loading model...
[2026-10-08 11:10:54,896] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 5.12MB/s]
model.safetensors.index.json: 70.1kB [00:00, 106MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:51<00:00, 17.11s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:07<00:00, 210MB/s]
Loading weights: 100%|█| 686/686 [00:26<00:00, 26.17it/s, Materializing param=mo
preprocessor_config.json: 100%|█████████████████| 505/505 [00:00<00:00, 786kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
tokenizer_config.json: 1.45kB [00:00, 4.68MB/s]
tokenizer.json: 3.62MB [00:00, 165MB/s]
toke

In [58]:
!mkdir -p outputs/checkpoints

In [60]:
!cp -r /kaggle/input/datasets/tebogoseopa20/stage1-best-v2 outputs/checkpoints/stage1_best_local

In [61]:
!cp -r /kaggle/input/datasets/tebogoseopa20/stage2-checkpoints-lmonly-best outputs/checkpoints/stage2_best_lmonly_local

In [62]:
!ls outputs/checkpoints/stage1_best_local

adapter_config.json	   processor_config.json  tokenizer_config.json
adapter_model.safetensors  README.md		  tokenizer.json
chat_template.jinja	   stage1-best-v2


In [63]:
!python scripts/calibrate_threshold.py --config configs/diag_stage1_only.yaml --blank-image

Loading model...
[2026-10-08 11:14:33,010] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 46603.38it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 165.29it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-08 11:14:41,397] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
>>> BLANK-IMAGE 

In [64]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_cosmos_scores.jsonl outputs/predictions/diag_stage1_only_blank_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
B = outputs/predictions/diag_stage1_only_blank_cosmos_scores.jsonl  (threshold 0.85)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.0139   95% CI [-0.0111, +0.0399]   p~0.293   (not significant)
Accuracy  A-B = +0.0273   95% CI [-0.0195, +0.0742]   p~0.286   (not significant)


In [65]:
%env SEARCH_API_KEY= 24f3f7b5b1644256e1a6b21fcf759e4da8d92b64

env: SEARCH_API_KEY=24f3f7b5b1644256e1a6b21fcf759e4da8d92b64


In [66]:
!grep -n "use_reranking\|rerank_keep_k\|n_queries" src/misinfodet/utils.py configs/diag_stage1_only.yaml

src/misinfodet/utils.py:73:    use_reranking: bool = True
src/misinfodet/utils.py:76:    rerank_keep_k: int = 3
configs/diag_stage1_only.yaml:9:use_reranking: true
configs/diag_stage1_only.yaml:13:rerank_keep_k: 3


In [67]:
!python scripts/calibrate_retrieval.py --config configs/diag_stage1_only.yaml

python3: can't open file '/kaggle/working/Honours-Research-Project/scripts/calibrate_retrieval.py': [Errno 2] No such file or directory


In [70]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.02 KiB | 2.02 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   97546ba..e5803cd  main       -> origin/main
Updating 97546ba..e5803cd
Fast-forward
 scripts/calibrate_retrieval.py | 99 ++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 99 insertions(+)
 create mode 100644 scripts/calibrate_retrieval.py


In [71]:
!ls scripts/calibrate_retrieval.py

scripts/calibrate_retrieval.py


In [72]:
!python scripts/calibrate_retrieval.py --config configs/diag_stage1_only.yaml

0 already scored, 510 to go
reranking=on, provider=serper
Loading model...
[2026-10-08 11:38:26,046] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 17026.94it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 167.86it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-08 11:38:34,258] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations

In [74]:
!mkdir -p /kaggle/working/pred_ds && cp outputs/predictions/*.jsonl outputs/predictions/*.json /kaggle/working/pred_ds/

In [80]:
import json
json.dump({"title": "misinfodet-predictions", "id": "tebogoseopa20/misinfodet-predictions", "licenses": [{"name": "CC0-1.0"}]}, open("/kaggle/working/pred_ds/dataset-metadata.json", "w"))

In [82]:
!kaggle datasets version -p /kaggle/working/pred_ds -m "add retrieval, leakage, stage1 blank"

Starting upload for file base_llava_eval_cosmos_scores.jsonl
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 147kB/s]
Upload successful: base_llava_eval_cosmos_scores.jsonl (38KB)
Starting upload for file stage2_eval_lmonly_mmfakebench_blank_scores.jsonl
100%|█████████████████████████████████████████| 105k/105k [00:00<00:00, 366kB/s]
Upload successful: stage2_eval_lmonly_mmfakebench_blank_scores.jsonl (105KB)
Starting upload for file diag_stage1_only_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 147kB/s]
Upload successful: diag_stage1_only_cosmos_scores.jsonl (38KB)
Starting upload for file diag_stage1_only_retrieval_cosmos_scores.jsonl
100%|████████████████████████████████████████| 696k/696k [00:00<00:00, 2.46MB/s]
Upload successful: diag_stage1_only_retrieval_cosmos_scores.jsonl (696KB)
Starting upload for fi

In [83]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.55 KiB | 1.55 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   e5803cd..6f69deb  main       -> origin/main
Updating e5803cd..6f69deb
Fast-forward
 scripts/rescore_retrieval.py | 67 ++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 67 insertions(+)
 create mode 100644 scripts/rescore_retrieval.py


In [84]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_cosmos_scores.jsonl outputs/predictions/diag_stage1_only_retrieval_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
B = outputs/predictions/diag_stage1_only_retrieval_cosmos_scores.jsonl  (threshold 0.87)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.1633   95% CI [+0.1027, +0.2263]   p~0.000   (significant)
Accuracy  A-B = +0.1523   95% CI [+0.0859, +0.2188]   p~0.000   (significant)


In [85]:
!python scripts/rescore_retrieval.py --config configs/diag_stage1_only.yaml

Loading model...
[2026-10-08 13:33:56,271] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 17672.63it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 164.03it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-08 13:34:04,608] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
  val 20/254 don

In [86]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl outputs/predictions/diag_stage1_only_retrieval_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl  (threshold 0.87)
B = outputs/predictions/diag_stage1_only_retrieval_cosmos_scores.jsonl  (threshold 0.87)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.1153   95% CI [+0.0596, +0.1740]   p~0.000   (significant)
Accuracy  A-B = +0.0664   95% CI [+0.0078, +0.1250]   p~0.030   (significant)


In [87]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_cosmos_scores.jsonl outputs/predictions/diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
B = outputs/predictions/diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl  (threshold 0.87)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.0480   95% CI [+0.0060, +0.0906]   p~0.026   (significant)
Accuracy  A-B = +0.0859   95% CI [+0.0234, +0.1445]   p~0.009   (significant)


In [88]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 599 bytes | 599.00 KiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   6f69deb..1b6c3c3  main       -> origin/main
Updating 6f69deb..1b6c3c3
Fast-forward
 scripts/rescore_retrieval.py | 17 ++++++++++-------
 1 file changed, 10 insertions(+), 7 deletions(-)


In [89]:
!python scripts/rescore_retrieval.py --config configs/diag_stage1_only.yaml --no-evidence

Loading model...
[2026-10-08 13:50:42,420] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 42653.94it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 167.26it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-08 13:50:50,674] INFO misinfodet.models.model_factory: Applying stage1 adapter from outputs/checkpoints/stage1_best_local
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
  val 20/254 don

In [93]:
!git pull

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 3.75 KiB | 3.75 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   51b94c7..722969b  main       -> origin/main
Updating 51b94c7..722969b
Fast-forward
 scripts/explain_verdicts.py  | 114 +++++++++++++++++++++++++++++++++++++
 scripts/rescore_retrieval.py | 130 ++++++++++++++-----------------------------
 2 files changed, 157 insertions(+), 87 deletions(-)
 create mode 100644 scripts/explain_verdicts.py


In [94]:
!ls scripts/explain_verdicts.py

scripts/explain_verdicts.py


In [95]:
!ls scripts/explain_verdicts.py

scripts/explain_verdicts.py


In [96]:
!python scripts/explain_verdicts.py

Verdict threshold (from validation): 0.81
Loading explanation model...
[2026-10-08 14:11:57,544] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 48770.98it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 169.06it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/explain_verdicts.py", line 114, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/explain_verdicts.py", line 62, in main
    model, processor = load_llava(cfg

In [97]:
p = "src/misinfodet/models/model_factory.py"
s = open(p).read()
new = '        if hasattr(model, "merge_adapter"):\n            model.merge_adapter()\n'
if new not in s:
    s = s.replace("        model.merge_adapter()\n", new, 1)
    open(p, "w").write(s)
print(s[s.index("if not trainable:"):s.index("if trainable:")])

if not trainable:
        # merge_adapter() merges every nested adapter's LoRA deltas into its
        # base layer IN PLACE, without unloading/removing the PeftModel
        # wrapper. merge_and_unload() (unload + remove) was tried first but
        # crashes on a doubly-nested PeftModel: it ends by deleting the
        # inner model's `peft_config`, which is a property with no deleter.
        # merge_adapter() walks every submodule regardless of nesting depth,
        # so one call merges both stage1 and stage2 in a single pass.
        if hasattr(model, "merge_adapter"):
            model.merge_adapter()

    


In [98]:
!python scripts/explain_verdicts.py

Verdict threshold (from validation): 0.81
Loading explanation model...
[2026-10-08 14:13:11,920] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 5667.98it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 168.73it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  10/256 done
  20/256 done
  30/256 done
  40/256 done
  50/256 done
  60/256 done
  70/256 done
  80/256 done
  90/256 done
  100/256 done
  110

In [99]:
!cd outputs/explanations && zip -r /kaggle/working/rating_pack.zip rating_sheet.csv rating_images explanations_cosmos_test.jsonl

  adding: rating_sheet.csv (deflated 70%)
  adding: rating_images/ (stored 0%)
  adding: rating_images/30.jpg (deflated 4%)
  adding: rating_images/27.jpg (deflated 4%)
  adding: rating_images/26.png (deflated 7%)
  adding: rating_images/50.jpg (deflated 5%)
  adding: rating_images/38.jpg (deflated 4%)
  adding: rating_images/06.png (deflated 3%)
  adding: rating_images/41.png (deflated 0%)
  adding: rating_images/20.jpg (deflated 6%)
  adding: rating_images/29.jpg (deflated 3%)
  adding: rating_images/16.png (deflated 1%)
  adding: rating_images/11.jpg (deflated 3%)
  adding: rating_images/35.jpg (deflated 4%)
  adding: rating_images/37.png (deflated 1%)
  adding: rating_images/43.jpg (deflated 3%)
  adding: rating_images/46.png (deflated 10%)
  adding: rating_images/15.jpg (deflated 4%)
  adding: rating_images/02.jpg (deflated 3%)
  adding: rating_images/10.jpg (deflated 3%)
  adding: rating_images/42.jpg (deflated 4%)
  adding: rating_images/01.jpg (deflated 6%)
  adding: rating_ima

In [100]:
%cd /kaggle/working

/kaggle/working


In [101]:
from IPython.display import FileLink
FileLink("rating_pack.zip")

/kaggle/working/rating_pack.zip

In [103]:
!mkdir -p /kaggle/working/pred_ds && cp /kaggle/working/Honours-Research-Project/outputs/predictions/* /kaggle/working/pred_ds/ && cp /kaggle/working/rating_pack.zip /kaggle/working/pred_ds/

In [104]:
import json
json.dump({"title": "misinfodet-predictions", "id": "tebogoseopa20/misinfodet-predictions", "licenses": [{"name": "CC0-1.0"}]}, open("/kaggle/working/pred_ds/dataset-metadata.json", "w"))

In [105]:
!kaggle datasets version -p /kaggle/working/pred_ds -m "retrieval ablations, leakage, stage1 blank, explanations"

Starting upload for file base_llava_eval_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 141kB/s]
Upload successful: base_llava_eval_cosmos_scores.jsonl (38KB)
Starting upload for file diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 132kB/s]
Upload successful: diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl (38KB)
Starting upload for file stage2_eval_lmonly_mmfakebench_blank_scores.jsonl
100%|█████████████████████████████████████████| 105k/105k [00:00<00:00, 396kB/s]
Upload successful: stage2_eval_lmonly_mmfakebench_blank_scores.jsonl (105KB)
Starting upload for file diag_stage1_only_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 147kB/s]
Upload successful: diag_stage1_only_cosmos_scores.jsonl (38KB)
Starting upload for file diag_stage1_only_retrieval_cosmos_scores.jsonl
100%|████████████████████████████████████████

In [106]:
!kaggle datasets files tebogoseopa20/misinfodet-predictions

Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U47zbycwTakoqC-sAy14ucIQNlur86-fn7paSQVs5b11pqYQxf4dwZcpRO7E_4dpvtEbWk5guDbHw63de__X539B83mMebSpR1qamKrApjFxK2AmtPxz0jGSXTPV40m71Cq-7i0aDNtcf4
name                                                            size  creationDate                
------------------------------------------------------------  ------  --------------------------  
base_llava_eval_cosmos_scores.jsonl                            38476  2026-10-08 14:54:23.035000  
base_llava_eval_mmfakebench_scores.jsonl                      107522  2026-10-08 14:54:23.050000  
diag_stage1_only_blank_cosmos_scores.jsonl                     38471  2026-10-08 14:54:23.059000  
diag_stage1_only_cosmos_scores.jsonl                           38479  2026-10-08 14:54:23.024000  
diag_stage1_only_mmfakebench_scores.jsonl                     107550  2026-10-08 14:54:23.045000  
diag_stage1_only_retrieval_cosmos_scores.jsonl                712327  2026-10-08 14:54:23.107000  
diag_stage1_only_retri

In [107]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [108]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl outputs/predictions/diag_stage1_only_retrieval_template_only_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl  (threshold 0.87)
B = outputs/predictions/diag_stage1_only_retrieval_template_only_cosmos_scores.jsonl  (threshold 0.86)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = -0.0108   95% CI [-0.0512, +0.0307]   p~0.619   (not significant)
Accuracy  A-B = -0.0508   95% CI [-0.1094, +0.0117]   p~0.124   (not significant)


In [109]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.91 KiB | 2.91 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   722969b..59dd0f9  main       -> origin/main
Updating 722969b..59dd0f9
Fast-forward
 scripts/final_checks.py | 140 ++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 140 insertions(+)
 create mode 100644 scripts/final_checks.py


In [110]:
!python scripts/final_checks.py coverage

val: items 254 | with evidence 253 | snippets/item 2.99 | evidence words/item: mean 95.0, min 0, max 122 | items under 20 words 1
test: items 256 | with evidence 256 | snippets/item 3.00 | evidence words/item: mean 94.9, min 70, max 120 | items under 20 words 0

Example (test_0):
queries: ['London Brexit march December 2018', 'Pro-Brexit demonstration December 2018', 'Brexiteers eager to leave EU migration flow']
- Brexit | The electorate voted to leave the EU by a slight margin, with a 51.9% share of the vote, all regions of England and Wales except Greater London voting in favour ...
- See the Brexit march in London from above | Aerial video with no audio shows pro-European Union supporters marching in London June 23 to urge the government to give Britons a vote on a ...
- Anti-racist marchers in London claim victory over far-right ... | Anti-racist campaigners have said their protest rally was more than twice as popular as a pro-Brexit demonstration led by Ukip and the


In [111]:
!pip install -q sentence-transformers

In [112]:
!python scripts/final_checks.py sbert

config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 676kB/s]
README.md: 11.6kB [00:00, 28.3MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 277kB/s]
config.json: 100%|█████████████████████████████| 571/571 [00:00<00:00, 2.98MB/s]
model.safetensors: 100%|██████████████████████| 438M/438M [00:02<00:00, 197MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1223.77it/s, Materializing param=
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 363/363 [00:00<00:00, 2.39MB/s]
vocab.txt: 232kB [00:00, 40.8MB/s]
tokenizer.json: 466kB [00:00, 117MB/s]
config.json: 100%|█████████████████████████████| 190/190 [00:00<00:00, 1.10MB/s]
SBER

In [113]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_cosmos_scores.jsonl outputs/predictions/sbert_caption_sim_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
B = outputs/predictions/sbert_caption_sim_cosmos_scores.jsonl  (threshold 0.36)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.2354   95% CI [+0.1684, +0.3030]   p~0.000   (significant)
Accuracy  A-B = +0.1914   95% CI [+0.1211, +0.2617]   p~0.000   (significant)


In [114]:
!python scripts/final_checks.py explanations

explanations: 256
refer to visual content: 230 (89.8%)
possible self-contradiction (wording supports the opposite verdict only): 4 (1.6%)
  correct verdicts: 213 items, 3 flagged
  wrong verdicts: 43 items, 1 flagged
flagged ids (check by hand): ['test_30', 'test_34', 'test_71', 'test_242']

wrote rating_sheet_blind.csv (50 items), rating_sheet_rater2.csv (20 items), rating_key.csv (do not open until rating is finished)


In [115]:
!cd outputs/explanations && zip -r /kaggle/working/rating_pack_blind.zip rating_sheet_blind.csv rating_sheet_rater2.csv rating_key.csv rating_images

  adding: rating_sheet_blind.csv (deflated 70%)
  adding: rating_sheet_rater2.csv (deflated 66%)
  adding: rating_key.csv (deflated 80%)
  adding: rating_images/ (stored 0%)
  adding: rating_images/30.jpg (deflated 4%)
  adding: rating_images/27.jpg (deflated 4%)
  adding: rating_images/26.png (deflated 7%)
  adding: rating_images/50.jpg (deflated 5%)
  adding: rating_images/38.jpg (deflated 4%)
  adding: rating_images/06.png (deflated 3%)
  adding: rating_images/41.png (deflated 0%)
  adding: rating_images/20.jpg (deflated 6%)
  adding: rating_images/29.jpg (deflated 3%)
  adding: rating_images/16.png (deflated 1%)
  adding: rating_images/11.jpg (deflated 3%)
  adding: rating_images/35.jpg (deflated 4%)
  adding: rating_images/37.png (deflated 1%)
  adding: rating_images/43.jpg (deflated 3%)
  adding: rating_images/46.png (deflated 10%)
  adding: rating_images/15.jpg (deflated 4%)
  adding: rating_images/02.jpg (deflated 3%)
  adding: rating_images/10.jpg (deflated 3%)
  adding: ratin

In [116]:
%cd /kaggle/working

/kaggle/working


In [117]:
from IPython.display import FileLink
FileLink("rating_pack_blind.zip")

/kaggle/working/rating_pack_blind.zip

In [119]:
!mkdir -p /kaggle/working/pred_ds && cp /kaggle/working/Honours-Research-Project/outputs/predictions/* /kaggle/working/pred_ds/ && cp /kaggle/working/Honours-Research-Project/outputs/explanations/explanations_cosmos_test.jsonl /kaggle/working/pred_ds/ && cp /kaggle/working/rating_pack_blind.zip /kaggle/working/pred_ds/

In [126]:
import json
json.dump({"title": "misinfodet-predictions", "id": "tebogoseopa20/misinfodet-predictions", "licenses": [{"name": "CC0-1.0"}]}, open("/kaggle/working/pred_ds/dataset-metadata.json", "w"))

In [127]:
!kaggle datasets version -p /kaggle/working/pred_ds -m "final: all scores, leakage, retrieval, sbert, explanations"

Starting upload for file base_llava_eval_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 149kB/s]
Upload successful: base_llava_eval_cosmos_scores.jsonl (38KB)
Starting upload for file sbert_caption_sim_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.7k/37.7k [00:00<00:00, 143kB/s]
Upload successful: sbert_caption_sim_cosmos_scores.jsonl (38KB)
Starting upload for file diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/37.6k [00:00<00:00, 144kB/s]
Upload successful: diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl (38KB)
Starting upload for file stage2_eval_lmonly_mmfakebench_blank_scores.jsonl
100%|█████████████████████████████████████████| 105k/105k [00:00<00:00, 396kB/s]
Upload successful: stage2_eval_lmonly_mmfakebench_blank_scores.jsonl (105KB)
Starting upload for file diag_stage1_only_cosmos_scores.jsonl
100%|███████████████████████████████████████| 37.6k/3

In [129]:
!kaggle datasets files tebogoseopa20/misinfodet-predictions

Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U7qaWn-pp1gjkGmtE0pOm5qwwr0xe-PJOvBIhEyAUsC1QLeqFGTg-iIhoKBmZ5a_OxlqvuQvGN-cSn7zzRStyBQ5fDey-fyp2ntaygesvETST3n5CYS-D7cvY8lVs1OQ5GFBTM1qgzNcsE
name                                                            size  creationDate                
------------------------------------------------------------  ------  --------------------------  
base_llava_eval_cosmos_scores.jsonl                            38476  2026-10-08 15:35:02.328000  
base_llava_eval_mmfakebench_scores.jsonl                      107522  2026-10-08 15:35:02.428000  
diag_stage1_only_blank_cosmos_scores.jsonl                     38471  2026-10-08 15:35:02.385000  
diag_stage1_only_cosmos_scores.jsonl                           38479  2026-10-08 15:35:02.402000  
diag_stage1_only_mmfakebench_scores.jsonl                     107550  2026-10-08 15:35:02.353000  
diag_stage1_only_retrieval_cosmos_scores.jsonl                712327  2026-10-08 15:35:02.375000  
diag_stage1_only_retri

In [130]:
!kaggle datasets files tebogoseopa20/misinfodet-predictions --page-size 200 | grep -v "rating_images"

name                                                             size  creationDate                
------------------------------------------------------------  -------  --------------------------  
base_llava_eval_cosmos_scores.jsonl                             38476  2026-10-08 15:35:02.328000  
base_llava_eval_mmfakebench_scores.jsonl                       107522  2026-10-08 15:35:02.428000  
diag_stage1_only_blank_cosmos_scores.jsonl                      38471  2026-10-08 15:35:02.385000  
diag_stage1_only_cosmos_scores.jsonl                            38479  2026-10-08 15:35:02.402000  
diag_stage1_only_mmfakebench_scores.jsonl                      107550  2026-10-08 15:35:02.353000  
diag_stage1_only_retrieval_cosmos_scores.jsonl                 712327  2026-10-08 15:35:02.375000  
diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl           38483  2026-10-08 15:35:02.383000  
diag_stage1_only_retrieval_template_only_cosmos_scores.jsonl    38454  2026-10-08 15:35:02.339000  


In [17]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 3.68 KiB | 3.68 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   7eb6964..0e05b03  main       -> origin/main
Updating 7eb6964..0e05b03
Fast-forward
 scripts/clip_baseline.py | 212 +++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 212 insertions(+)
 create mode 100644 scripts/clip_baseline.py


In [18]:
!python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml --smoke 5

Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/hierarchical_mmfakebench.py", line 271, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/hierarchical_mmfakebench.py", line 243, in main
    records = read_jsonl(Path(cfg.data_dir) / "mmfakebench_test.jsonl")
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/utils.py", line 164, in read_jsonl
    with open(path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/mmfakebench_test.jsonl'


In [19]:
!ls /kaggle/input

datasets


In [20]:
!ls /kaggle/input/datasets/*

cosmos-dataset		stage1-best-v2		stage2-checkpoints-lmonly-best
misinfodet-predictions	stage2-best		stage2-checkpoints-resumed-best
mmfakebench		stage2-best-3ep-manual


In [21]:
!find /kaggle/input/datasets -maxdepth 4 -iname "*mmfake*" | head -20

/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/stage2_eval_lmonly_mmfakebench_blank_scores.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/diag_stage1_only_mmfakebench_scores.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/base_llava_eval_mmfakebench_scores.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl
/kaggle/input/datasets/tebogoseopa20/mmfakebench
/kaggle/input/datasets/tebogoseopa20/mmfakebench/MMFakeBench_test
/kaggle/input/datasets/tebogoseopa20/mmfakebench/MMFakeBench_test.json
/kaggle/input/datasets/tebogoseopa20/mmfakebench/MMFakeBench_val
/kaggle/input/datasets/tebogoseopa20/mmfakebench/MMFakeBench_val.json


In [22]:
!ls scripts | grep -i -E "mmfake|prepare|download"

calibrate_mmfakebench.py
hierarchical_mmfakebench.py
prepare_cosmos.py
prepare_mmfakebench.py
prepare_newsclippings.py


In [23]:
!ls data

ls: cannot access 'data': No such file or directory


In [24]:
!python scripts/prepare_mmfakebench.py --help

usage: prepare_mmfakebench.py [-h] [--config CONFIG]

options:
  -h, --help       show this help message and exit
  --config CONFIG


In [25]:
!grep -n -i -E "path|dir|hf_hub|snapshot|download|token|json" scripts/prepare_mmfakebench.py | head -30

2:"""Prepare MMFakeBench test split into unified JSONL (evaluation only)."""
4:from pathlib import Path
15:    n = prepare(cfg.mmfakebench_dir, Path(cfg.data_dir) / "mmfakebench_test.jsonl")


In [26]:
!grep -rn "mmfakebench_dir" configs/ src/ | head

grep: src/misinfodet/__pycache__/utils.cpython-312.pyc: binary file matches
configs/base.yaml:11:mmfakebench_dir: data/mmfakebench
grep: src/misinfodet/__pycache__/utils.cpython-311.pyc: binary file matches
grep: src/misinfodet/data/__pycache__/mmfakebench.cpython-312.pyc: binary file matches
src/misinfodet/utils.py:39:    mmfakebench_dir: str = "data/mmfakebench"
src/misinfodet/data/mmfakebench.py:27:    mmfakebench_dir: str | Path,
src/misinfodet/data/mmfakebench.py:32:    root = Path(mmfakebench_dir)


In [27]:
!sed -n 1,60p src/misinfodet/data/mmfakebench.py

"""MMFakeBench preparation (secondary benchmark, evaluation only).

Expected raw layout after download (see docs/DATASETS.md):

    data/mmfakebench/MMFakeBench_test/...      (images)
    data/mmfakebench/MMFakeBench_test.json     (annotations)

Each raw annotation has: image_path, text, gt_answers ("True"/"Fake"),
fake_cls (one of 12 sub-types, e.g. "original", "mismatch", "text_veracity...").

Unified schema mirrors NewsCLIPpings, with the fine-grained forgery type kept
for the mixed-source breakdown analysis in the final report:
    {"id", "image_path", "caption", "label", "forgery_type", "source"}
label: 0 = real, 1 = fake (any forgery source).
"""
from __future__ import annotations

import json
from pathlib import Path

from ..utils import get_logger, write_jsonl

log = get_logger(__name__)


def prepare(
    mmfakebench_dir: str | Path,
    out_path: str | Path,
    split: str = "test",
    max_samples: int | None = None,
) -> int:
    root = Path(mmfakebench_dir)
    ann_path = 

In [29]:
!grep -n -i -A 15 "mmfakebench" docs/DATASETS.md | head -40

37:The test set's shape also differs from NewsCLIPpings/MMFakeBench: each
38-image comes with **two** candidate captions and one label for whether
39-pairing them together is out-of-context, not one caption with its own
40-label. See `src/misinfodet/data/cosmos.py` docstring for the exact schema
41-this produces, and note that consuming it needs a two-caption prompt
42-template, not the existing single-caption one.
43-
44-## NewsCLIPpings (superseded — kept for reference only)
45-
46-VisualNews' image archive (`origin.tar`, 91GB) proved impractical to
47-download reliably (multiple corruption incidents over several days); the
48-project switched to COSMOS above. This section is left for reference in
49-case NewsCLIPpings is revisited later.
50-
51-NewsCLIPpings distributes only annotation files; images and captions come from
52-VisualNews, which is gated behind an access request.
--
73:## MMFakeBench (secondary, evaluation only)
74-
75:Download from the official release: https://github

In [30]:
!ls /kaggle/input/datasets/*

cosmos-dataset		stage1-best-v2		stage2-checkpoints-lmonly-best
misinfodet-predictions	stage2-best		stage2-checkpoints-resumed-best
mmfakebench		stage2-best-3ep-manual


In [31]:
!ls /kaggle/input/datasets/*/mmfakebench

MMFakeBench_test  MMFakeBench_test.json  MMFakeBench_val  MMFakeBench_val.json


In [32]:
!mkdir -p data && ln -sfn $(ls -d /kaggle/input/datasets/*/mmfakebench) data/mmfakebench

In [33]:
!ls data/mmfakebench

MMFakeBench_test  MMFakeBench_test.json  MMFakeBench_val  MMFakeBench_val.json


In [34]:
!python scripts/prepare_mmfakebench.py --config configs/base_llava_eval.yaml

[2026-10-15 12:04:37,076] INFO misinfodet.data.mmfakebench: MMFakeBench test: wrote 10000 records to data/mmfakebench_test.jsonl
Prepared 10000 MMFakeBench records


In [35]:
!python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml --smoke 5

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-15 12:05:18,995] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 3.96MB/s]
model.safetensors.index.json: 70.1kB [00:00, 113MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [01:00<00:00, 20.13s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:07<00:00, 208MB/s]
Loading weights: 100%|█| 686/686 [00:21<00:00, 32.65it/s, Materializing param=mo
preprocessor_config.json: 100%|████████████████| 505/505 [00:00<00:00, 2.18MB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow proces

In [36]:
import re, pathlib
p = pathlib.Path("src/misinfodet/models/model_factory.py")
lines = p.read_text().split("\n")
out = []
for i, l in enumerate(lines):
    m = re.match(r"^(\s*)model\.merge_adapter\(\)\s*$", l)
    if m and "hasattr" not in lines[i - 1]:
        out += [f'{m.group(1)}if hasattr(model, "merge_adapter"):', f"{m.group(1)}    model.merge_adapter()"]
    else:
        out.append(l)
p.write_text("\n".join(out))
print("\n".join(out[70:85]))

        # merge_adapter() merges every nested adapter's LoRA deltas into its
        # base layer IN PLACE, without unloading/removing the PeftModel
        # wrapper. merge_and_unload() (unload + remove) was tried first but
        # crashes on a doubly-nested PeftModel: it ends by deleting the
        # inner model's `peft_config`, which is a property with no deleter.
        # merge_adapter() walks every submodule regardless of nesting depth,
        # so one call merges both stage1 and stage2 in a single pass.
        if hasattr(model, "merge_adapter"):
            model.merge_adapter()

    if trainable:
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        if cfg.load_in_4bit:
            model = prepare_model_for_kbit_training(model)


In [37]:
!python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml --smoke 5

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-15 12:13:37,891] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 22349.75it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 159.10it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
0 already scored, 1 to go
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/hierarchical_mmfakebench.py", line 271, in <module>
    main()
  File "/

In [38]:
!ls /kaggle/input/datasets/*/mmfakebench/MMFakeBench_test

fake  real  source


In [41]:
!rm data/mmfakebench && mkdir -p data/mmfakebench

rm: cannot remove 'data/mmfakebench': Is a directory


In [49]:
!rm -f data/mmfakebench/mmfakebench

In [50]:
!SRC=$(ls -d /kaggle/input/datasets/*/mmfakebench); ln -sfn $SRC/MMFakeBench_test.json data/mmfakebench/; for d in $SRC/MMFakeBench_test/*; do ln -sfn $d data/mmfakebench/; done; ls data/mmfakebench

fake  MMFakeBench_test.json  real  source


In [51]:
!ls data/mmfakebench/fake/Newsclipings_person_test_500/Newsclipings_person_test_262.png

data/mmfakebench/fake/Newsclipings_person_test_500/Newsclipings_person_test_262.png


In [52]:
!python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml --smoke 5

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-15 12:17:43,127] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 45100.04it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 159.34it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
0 already scored, 1 to go
mismatch {'p_text_false': 0.3007446825504303, 'p_visual_fake': 0.29421451687812805, 'p_inconsistent': 0.392337441444397, 'p_standard': [0.43847036361694336, 0.21369

In [53]:
!find /kaggle/input/datasets/*/misinfodet-predictions -name "*mmfakebench_scores.jsonl"

/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/diag_stage1_only_mmfakebench_scores.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/base_llava_eval_mmfakebench_scores.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/stage2_eval_lmonly_mmfakebench_scores.jsonl


In [54]:
!mkdir -p outputs/predictions && cp $(find /kaggle/input/datasets/*/misinfodet-predictions -name "*mmfakebench_scores.jsonl") outputs/predictions/

In [55]:
!ls outputs/predictions

base_llava_eval_mmfakebench_hierarchical_smoke.jsonl
base_llava_eval_mmfakebench_scores.jsonl
diag_stage1_only_mmfakebench_scores.jsonl
stage2_eval_lmonly_mmfakebench_scores.jsonl


In [56]:
!python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml --baseline-scores outputs/predictions/base_llava_eval_mmfakebench_scores.jsonl

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-15 12:25:08,103] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 26658.71it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 158.97it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
0 already scored, 2000 to go
  20/2000  (2.7s/item, ~89 min left)
  40/2000  (2.8s/item, ~93 min left)
  60/2000  (3.0s/item, ~96 min left)
  80/2000  (3.1s/item, ~98 min left)
  100/2000  (

In [61]:
   !cp outputs/predictions/base_llava_eval_mmfakebench_hierarchical.jsonl /kaggle/working/

In [62]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.90 KiB | 1.90 MiB/s, done.
From https://github.com/TebogoSeopa20/Honours-Research-Project
   7f4c02c..d4407e8  main       -> origin/main
Updating 7f4c02c..d4407e8
Fast-forward
 scripts/fuse_mmfakebench.py | 88 +++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 88 insertions(+)
 create mode 100644 scripts/fuse_mmfakebench.py


In [63]:
!python scripts/fuse_mmfakebench.py --config configs/base_llava_eval.yaml

Run: base_llava_eval  calibration n=1000  evaluation n=1000

=== Binary real vs fake, logistic fusion fitted on calibration half (evaluation-half AUROC) ===
features                          AUROC   vs single verdict
single verdict only              0.679
three checks only                0.650   -0.029  95% CI [-0.054, -0.003]  p=0.027
single + three checks            0.673   -0.006  95% CI [-0.024, +0.013]  p=0.538
   weights: single=+3.01  p_text_false=-0.13  p_visual_fake=+1.70  p_inconsistent=+1.52
single + three checks + 4-way    0.730   +0.051  95% CI [+0.027, +0.076]  p=0.000

=== Per type vs real (AUROC): single verdict -> fused (single + three checks) ===
textual   0.803 -> 0.737
visual    0.638 -> 0.785
mismatch  0.568 -> 0.572

=== 4-class, multinomial logistic fusion fitted on calibration half ===
macro F1=55.3  accuracy=55.4  per-class F1: real=50.7  textual=63.7  visual=56.1  mismatch=50.7


In [64]:
!grep -ln "stage1_adapter" configs/*.yaml

configs/base_llava_eval.yaml
configs/diag_stage1_only.yaml
configs/stage2_eval_3ep.yaml
configs/stage2_eval_lmonly.yaml
configs/stage2_eval_resumed.yaml
configs/stage2_eval.yaml
configs/stage2_ooc_finetune_lmonly.yaml
configs/stage2_ooc_finetune.yaml
configs/stage2_resume.yaml


In [65]:
!grep -n "run_name\|stage1_adapter\|stage2_adapter" $(grep -ln "stage1_adapter" configs/*.yaml)

configs/base_llava_eval.yaml:1:run_name: base_llava_eval
configs/base_llava_eval.yaml:5:stage1_adapter: null
configs/base_llava_eval.yaml:6:stage2_adapter: null
configs/diag_stage1_only.yaml:1:run_name: diag_stage1_only
configs/diag_stage1_only.yaml:5:stage1_adapter: outputs/checkpoints/stage1_best_local
configs/diag_stage1_only.yaml:6:stage2_adapter: ""
configs/stage2_eval_3ep.yaml:1:run_name: stage2_eval_3ep
configs/stage2_eval_3ep.yaml:5:stage1_adapter: outputs/checkpoints/stage1_best_local
configs/stage2_eval_3ep.yaml:6:stage2_adapter: outputs/checkpoints/stage2_best_3ep_local
configs/stage2_eval_lmonly.yaml:1:run_name: stage2_eval_lmonly
configs/stage2_eval_lmonly.yaml:5:stage1_adapter: outputs/checkpoints/stage1_best_local
configs/stage2_eval_lmonly.yaml:6:stage2_adapter: outputs/checkpoints/stage2_best_lmonly_local
configs/stage2_eval_resumed.yaml:1:run_name: stage2_eval_resumed
configs/stage2_eval_resumed.yaml:5:stage1_adapter: outputs/checkpoints/stage1_best_local
configs/stag

In [66]:
!ls /kaggle/input/datasets/*/stage1-best-v2

adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json


In [70]:
!cp configs/base_llava_eval.yaml configs/diag_stage1_only.yaml

In [73]:
!sed -i "s|^run_name:.*|run_name: diag_stage1_only|; s|^stage1_adapter:.*|stage1_adapter: $(ls -d /kaggle/input/datasets/*/stage1-best-v2)|" configs/diag_stage1_only.yaml

In [74]:
!cat configs/diag_stage1_only.yaml

run_name: diag_stage1_only
seed: 42
base_model: llava-hf/llava-1.5-7b-hf
load_in_4bit: true
stage1_adapter: /kaggle/input/datasets/tebogoseopa20/stage1-best-v2
stage2_adapter: null
wandb: false
max_samples: null


In [75]:
!ls outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl

outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl


In [ ]:
!python scripts/hierarchical_mmfakebench.py --config configs/diag_stage1_only.yaml --baseline-scores outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-15 14:32:36,619] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 25784.66it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 156.40it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-15 14:32:51,398] INFO misinfodet.models.model_factory: Applying stage1 adapter from /kaggle/input/datasets/tebogoseopa20/stage1-best-v2
/usr/local/lib/python3.12/dist-packages/peft/

In [ ]:
import os, re, pathlib, subprocess

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash",
                   env={**os.environ, "PYTHONUNBUFFERED": "1"})

os.chdir("/kaggle/working")
if not os.path.isdir("Honours-Research-Project"):
    sh("git clone https://github.com/TebogoSeopa20/Honours-Research-Project.git")
os.chdir("/kaggle/working/Honours-Research-Project")

p = pathlib.Path("src/misinfodet/models/model_factory.py")
lines = p.read_text().split("\n")
out = []
for i, l in enumerate(lines):
    m = re.match(r"^(\s*)model\.merge_adapter\(\)\s*$", l)
    if m and "hasattr" not in lines[i - 1]:
        out += [f'{m.group(1)}if hasattr(model, "merge_adapter"):', f"{m.group(1)}    model.merge_adapter()"]
    else:
        out.append(l)
p.write_text("\n".join(out))

sh('mkdir -p data/mmfakebench && SRC=$(ls -d /kaggle/input/datasets/*/mmfakebench); '
   'ln -sfn $SRC/MMFakeBench_test.json data/mmfakebench/; '
   'for d in $SRC/MMFakeBench_test/*; do ln -sfn $d data/mmfakebench/; done; ls data/mmfakebench')
sh("python scripts/prepare_mmfakebench.py --config configs/base_llava_eval.yaml")
sh('mkdir -p outputs/predictions && find /kaggle/input/datasets/*/misinfodet-predictions '
   '\\( -name "*mmfakebench_scores.jsonl" -o -name "*mmfakebench_hierarchical.jsonl" \\) '
   '-exec cp -n {} outputs/predictions/ \\; ; ls -la outputs/predictions')
sh('cp configs/base_llava_eval.yaml configs/diag_stage1_only.yaml && '
   'sed -i "s|^run_name:.*|run_name: diag_stage1_only|; '
   's|^stage1_adapter:.*|stage1_adapter: $(ls -d /kaggle/input/datasets/*/stage1-best-v2)|" '
   'configs/diag_stage1_only.yaml && cat configs/diag_stage1_only.yaml')

sh("python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml "
   "--baseline-scores outputs/predictions/base_llava_eval_mmfakebench_scores.jsonl")
sh("python scripts/fuse_mmfakebench.py --config configs/base_llava_eval.yaml")
sh("cp outputs/predictions/base_llava_eval_mmfakebench_hierarchical.jsonl /kaggle/working/")
sh("python scripts/hierarchical_mmfakebench.py --config configs/diag_stage1_only.yaml "
   "--baseline-scores outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl")
sh("python scripts/fuse_mmfakebench.py --config configs/diag_stage1_only.yaml")
sh("cp outputs/predictions/*_hierarchical.jsonl /kaggle/working/")


$ mkdir -p data/mmfakebench && SRC=$(ls -d /kaggle/input/datasets/*/mmfakebench); ln -sfn $SRC/MMFakeBench_test.json data/mmfakebench/; for d in $SRC/MMFakeBench_test/*; do ln -sfn $d data/mmfakebench/; done; ls data/mmfakebench
fake
MMFakeBench_test.json
real
source

$ python scripts/prepare_mmfakebench.py --config configs/base_llava_eval.yaml
Prepared 10000 MMFakeBench records

$ mkdir -p outputs/predictions && find /kaggle/input/datasets/*/misinfodet-predictions \( -name "*mmfakebench_scores.jsonl" -o -name "*mmfakebench_hierarchical.jsonl" \) -exec cp -n {} outputs/predictions/ \; ; ls -la outputs/predictions
total 1028
drwxr-xr-x 2 root root   4096 Oct 16 07:25 .
drwxr-xr-x 3 root root   4096 Oct 16 07:23 ..
-rw-r--r-- 1 root root 274920 Oct 16 08:27 base_llava_eval_mmfakebench_hierarchical.jsonl
-rw-r--r-- 1 root root 107522 Oct 16 07:23 base_llava_eval_mmfakebench_scores.jsonl
-rw-r--r-- 1 root root 107550 Oct 16 07:23 diag_stage1_only_mmfakebench_scores.jsonl
-rw-r--r-- 1 root

In [16]:
!git clone https://github.com/TebogoSeopa20/Honours-Research-Project.git

Cloning into 'Honours-Research-Project'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 327 (delta 172), reused 250 (delta 97), pack-reused 0 (from 0)
Receiving objects: 100% (327/327), 315.24 KiB | 2.03 MiB/s, done.
Resolving deltas: 100% (172/172), done.


In [2]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [3]:
import re, pathlib
p = pathlib.Path("src/misinfodet/models/model_factory.py")
lines = p.read_text().split("\n")
out = []
for i, l in enumerate(lines):
    m = re.match(r"^(\s*)model\.merge_adapter\(\)\s*$", l)
    if m and "hasattr" not in lines[i - 1]:
        out += [f'{m.group(1)}if hasattr(model, "merge_adapter"):', f"{m.group(1)}    model.merge_adapter()"]
    else:
        out.append(l)
p.write_text("\n".join(out))

10141

In [4]:
!mkdir -p data/mmfakebench && SRC=$(ls -d /kaggle/input/datasets/*/mmfakebench); ln -sfn $SRC/MMFakeBench_test.json data/mmfakebench/; for d in $SRC/MMFakeBench_test/*; do ln -sfn $d data/mmfakebench/; done; ls data/mmfakebench

fake  MMFakeBench_test.json  real  source


In [6]:
%env PYTHONPATH=/kaggle/working/Honours-Research-Project/src

env: PYTHONPATH=/kaggle/working/Honours-Research-Project/src


In [7]:
!python scripts/prepare_mmfakebench.py --config configs/base_llava_eval.yaml

[2026-10-17 15:46:06,131] INFO misinfodet.data.mmfakebench: MMFakeBench test: wrote 10000 records to data/mmfakebench_test.jsonl
Prepared 10000 MMFakeBench records


In [8]:
!mkdir -p outputs/predictions && cp $(find /kaggle/input/datasets/*/misinfodet-predictions -name "*mmfakebench_scores.jsonl" -o -name "*mmfakebench_hierarchical.jsonl") outputs/predictions/; ls outputs/predictions

base_llava_eval_mmfakebench_scores.jsonl
diag_stage1_only_mmfakebench_scores.jsonl
stage2_eval_lmonly_mmfakebench_scores.jsonl


In [9]:
!cp configs/base_llava_eval.yaml configs/diag_stage1_only.yaml && sed -i "s|^run_name:.*|run_name: diag_stage1_only|; s|^stage1_adapter:.*|stage1_adapter: $(ls -d /kaggle/input/datasets/*/stage1-best-v2)|" configs/diag_stage1_only.yaml && cat configs/diag_stage1_only.yaml

run_name: diag_stage1_only
seed: 42
base_model: llava-hf/llava-1.5-7b-hf
load_in_4bit: true
stage1_adapter: /kaggle/input/datasets/tebogoseopa20/stage1-best-v2
stage2_adapter: null
wandb: false
max_samples: null


In [10]:
!python scripts/hierarchical_mmfakebench.py --config configs/diag_stage1_only.yaml --baseline-scores outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-17 15:47:53,650] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 5.61MB/s]
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/hierarchical_mmfakebench.py", line 271, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/hierarchical_mmfakebench.py", line 257, in main
    model, processor = load_llava(cfg, trainable=False)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/models/model_factory.py", line 46, in load_llava
    model = LlavaForConditionalGeneration.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/d

In [11]:
!pip install -q -U "bitsandbytes>=0.46.1" peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.8 MB/s eta 0:00:00


In [12]:
!python scripts/hierarchical_mmfakebench.py --config configs/base_llava_eval.yaml --baseline-scores outputs/predictions/base_llava_eval_mmfakebench_scores.jsonl

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-17 15:50:28,420] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
model.safetensors.index.json: 70.1kB [00:00, 124MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [01:01<00:00, 20.43s/it]
Download complete: 100%|████████████████████| 14.1G/14.1G [01:07<00:00, 210MB/s]
Loading weights: 100%|█| 686/686 [00:15<00:00, 45.61it/s, Materializing param=mo
preprocessor_config.json: 100%|█████████████████| 505/505 [00:00<00:00, 861kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
tokenizer_config.json: 1.45kB

In [13]:
!cp outputs/predictions/base_llava_eval_mmfakebench_hierarchical.jsonl /kaggle/working/

In [14]:
!python scripts/fuse_mmfakebench.py --config configs/base_llava_eval.yaml

Run: base_llava_eval  calibration n=1000  evaluation n=1000

=== Binary real vs fake, logistic fusion fitted on calibration half (evaluation-half AUROC) ===
features                          AUROC   vs single verdict
single verdict only              0.679
three checks only                0.650   -0.029  95% CI [-0.054, -0.003]  p=0.027
single + three checks            0.673   -0.006  95% CI [-0.024, +0.013]  p=0.538
   weights: single=+3.01  p_text_false=-0.13  p_visual_fake=+1.70  p_inconsistent=+1.52
single + three checks + 4-way    0.730   +0.051  95% CI [+0.027, +0.076]  p=0.000

=== Per type vs real (AUROC): single verdict -> fused (single + three checks) ===
textual   0.803 -> 0.737
visual    0.638 -> 0.785
mismatch  0.568 -> 0.572

=== 4-class, multinomial logistic fusion fitted on calibration half ===
macro F1=55.3  accuracy=55.4  per-class F1: real=50.7  textual=63.7  visual=56.1  mismatch=50.7


In [ ]:
!python scripts/hierarchical_mmfakebench.py --config configs/diag_stage1_only.yaml --baseline-scores outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation
Loading model...
[2026-10-17 17:52:25,667] INFO misinfodet.models.model_factory: Loading base model llava-hf/llava-1.5-7b-hf (4bit=True)
Fetching 3 files: 100%|████████████████████████| 3/3 [00:00<00:00, 24385.49it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|█| 686/686 [00:04<00:00, 162.62it/s, Materializing param=m
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[2026-10-17 17:52:36,494] INFO misinfodet.models.model_factory: Applying stage1 adapter from /kaggle/input/datasets/tebogoseopa20/stage1-best-v2
/usr/local/lib/python3.12/dist-packages/peft/

In [16]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [17]:
%env PYTHONPATH=/kaggle/working/Honours-Research-Project/src

env: PYTHONPATH=/kaggle/working/Honours-Research-Project/src


In [18]:
!wc -l outputs/predictions/diag_stage1_only_mmfakebench_hierarchical.jsonl

2000 outputs/predictions/diag_stage1_only_mmfakebench_hierarchical.jsonl


In [19]:
!python scripts/hierarchical_mmfakebench.py --config configs/diag_stage1_only.yaml --baseline-scores outputs/predictions/diag_stage1_only_mmfakebench_scores.jsonl --analyse-only

forgery types: ['mismatch', 'original', 'textual_veracity_distortion', 'visual_veracity_distortion']
Sample: 1000 calibration + 1000 evaluation

Evaluation half: n=1000  real=300  textual=300  visual=100  mismatch=300

=== 4-class: standard prompting (zero-shot, argmax) ===
macro F1=39.3  accuracy=44.5  per-class F1: real=45.6  textual=56.5  visual=49.6  mismatch=5.6

=== 4-class: hierarchical (MMD-Agent-style), thresholds from calibration half: text 0.45, visual 0.55, consistency 0.45 ===
macro F1=46.4  accuracy=45.3  per-class F1: real=39.9  textual=41.3  visual=55.8  mismatch=48.4

=== Binary real vs fake AUROC (evaluation half) ===
standard 4-way prompt (1 - P(real)): 0.712
hierarchical (max of three checks):  0.679

=== Each check on its own target type vs real (AUROC) ===
p_text_false    textual   vs real: 0.750
p_visual_fake   visual    vs real: 0.974
p_inconsistent  mismatch  vs real: 0.630

=== Paired bootstrap vs single verdict prompt (diag_stage1_only_mmfakebench_scores.json

In [23]:
!cp /kaggle/working/Honours-Research-Project/outputs/predictions/diag_stage1_only_mmfakebench_hierarchical.jsonl /kaggle/working/

In [24]:
!python scripts/fuse_mmfakebench.py --config configs/diag_stage1_only.yaml

Run: diag_stage1_only  calibration n=1000  evaluation n=1000

=== Binary real vs fake, logistic fusion fitted on calibration half (evaluation-half AUROC) ===
features                          AUROC   vs single verdict
single verdict only              0.707
three checks only                0.696   -0.011  95% CI [-0.033, +0.011]  p=0.319
single + three checks            0.710   +0.004  95% CI [-0.013, +0.020]  p=0.662
   weights: single=+2.97  p_text_false=+0.65  p_visual_fake=+1.75  p_inconsistent=+2.26
single + three checks + 4-way    0.739   +0.032  95% CI [+0.012, +0.052]  p=0.004

=== Per type vs real (AUROC): single verdict -> fused (single + three checks) ===
textual   0.823 -> 0.778
visual    0.706 -> 0.815
mismatch  0.590 -> 0.608

=== 4-class, multinomial logistic fusion fitted on calibration half ===
macro F1=58.5  accuracy=57.8  per-class F1: real=51.0  textual=66.5  visual=62.9  mismatch=53.6


In [2]:
!git clone https://github.com/g-luo/news_clippings.git /kaggle/working/news_clippings

Cloning into '/kaggle/working/news_clippings'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 64 (delta 8), reused 8 (delta 5), pack-reused 46 (from 1)
Receiving objects: 100% (64/64), 20.81 KiB | 10.40 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [3]:
!cd /kaggle/working/news_clippings && bash download.sh

--2026-10-21 12:11:16--  https://huggingface.co/g-luo/news-clippings/resolve/main/data/merged_balanced/train.json
Resolving huggingface.co (huggingface.co)... 3.169.137.19, 3.169.137.111, 3.169.137.5, ...
Connecting to huggingface.co (huggingface.co)|3.169.137.19|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66e0982445daee0b34d020e1/fb59bc4135baa9d828294c49bcd27adfc073ed8a6f93d6dbbd13be1a96fd3a36?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27train.json%3B+filename%3D%22train.json%22%3B&user_id=public&response-content-type=application%2Fjson%3B+charset%3Dutf-8&X-Xet-Cas-Uid=public&xip=YzPMdiRnaN8&Expires=1792588276&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZlMDk4MjQ0NWRhZWUwYjM0ZDAyMGUxL2ZiNTliYzQxMzViYWE5ZDgyODI5NGM0OWJjZDI3YWRmYzA3M2VkOGE2ZjkzZDZkYmJkMTNiZTFhOTZmZDNhMzZcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2U

In [4]:
!cd /kaggle/working/news_clippings/embeddings && rm -rf facenet_embeddings places_resnet50 sbert_embeddings clip_image_embeddings/*train.pkl clip_text_embeddings/*train.pkl && du -sh /kaggle/working

/bin/bash: line 1: cd: /kaggle/working/news_clippings/embeddings: No such file or directory


In [5]:
!find /kaggle/working -maxdepth 4 -type d -name "embeddings"; find /kaggle/working -maxdepth 5 -type d -name "merged_balanced"

/kaggle/working/news_clippings/news_clippings/embeddings
/kaggle/working/news_clippings/news_clippings/data/merged_balanced


In [6]:
!cd /kaggle/working/news_clippings/news_clippings/embeddings && rm -rf facenet_embeddings places_resnet50 sbert_embeddings clip_image_embeddings/*train.pkl clip_text_embeddings/*train.pkl && du -sh /kaggle/working

489M	/kaggle/working


In [7]:
!python scripts/prepare_newsclippings_subset.py --probe 20

origin 0
origin/bbc 0
origin/bbc/images 0
origin/bbc/images/0038 0
origin/bbc/images/0038/392.jpg 30131
origin/bbc/images/0038/435.jpg 96594
origin/bbc/images/0038/847.jpg 73315
origin/bbc/images/0038/054.jpg 33705
origin/bbc/images/0038/722.jpg 134891
origin/bbc/images/0038/085.jpg 43928
origin/bbc/images/0038/343.jpg 25622
origin/bbc/images/0038/896.jpg 32026
origin/bbc/images/0038/449.jpg 60148
origin/bbc/images/0038/028.jpg 32998
origin/bbc/images/0038/498.jpg 851979
origin/bbc/images/0038/606.jpg 43493
origin/bbc/images/0038/267.jpg 69617
origin/bbc/images/0038/963.jpg 54166
origin/bbc/images/0038/511.jpg 35639
origin/bbc/images/0038/170.jpg 359780


In [8]:
!python -u scripts/prepare_newsclippings_subset.py --url https://www.cs.rice.edu/~vo9/visualnews/articles.tar.gz --list-json

usage: prepare_newsclippings_subset.py [-h] [--url URL]
                                       [--annotations ANNOTATIONS] [--out OUT]
                                       [--n-test N_TEST] [--n-val N_VAL]
                                       [--seed SEED] [--probe PROBE]
prepare_newsclippings_subset.py: error: unrecognized arguments: --list-json


In [14]:
%%writefile scripts/prepare_newsclippings_subset.py
import argparse
import json
import random
import tarfile
import time
import urllib.request
from pathlib import Path


def stream(url):
    return tarfile.open(fileobj=urllib.request.urlopen(url), mode="r|*")


def norm(p):
    p = p.lstrip("./")
    return p[len("origin/"):] if p.startswith("origin/") else p


def probe(url, k):
    for i, m in enumerate(stream(url)):
        print(m.name, m.size)
        if i + 1 >= k:
            break


def list_json(url):
    t0 = time.time()
    for i, m in enumerate(stream(url)):
        if m.isfile() and m.name.endswith(".json"):
            print(f"{m.name}  {m.size / 1e6:.1f} MB  (member {i + 1}, {time.time() - t0:.0f}s)", flush=True)
        if (i + 1) % 200000 == 0:
            print(f"  scanned {i + 1} members, {time.time() - t0:.0f}s", flush=True)
    print(f"done, {time.time() - t0:.0f}s")


def extract_one(url, suffix, dest):
    tf = stream(url)
    for m in tf:
        if m.isfile() and m.name.endswith(suffix):
            Path(dest).parent.mkdir(parents=True, exist_ok=True)
            Path(dest).write_bytes(tf.extractfile(m).read())
            print(f"saved {m.name} -> {dest}")
            return
    raise SystemExit(f"{suffix} not found")


def sample(anns, n, seed):
    rng = random.Random(seed)
    pos = [a for a in anns if a["falsified"]]
    neg = [a for a in anns if not a["falsified"]]
    k = n // 2
    return rng.sample(pos, min(k, len(pos))) + rng.sample(neg, min(n - k, len(neg)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", default="https://www.cs.rice.edu/~vo9/visualnews/origin.tar")
    ap.add_argument("--annotations", default="news_clippings/data/merged_balanced")
    ap.add_argument("--out", default="data/newsclippings")
    ap.add_argument("--n-test", type=int, default=1000)
    ap.add_argument("--n-val", type=int, default=500)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--probe", type=int, default=0)
    ap.add_argument("--list-json", action="store_true")
    ap.add_argument("--extract", default=None)
    args = ap.parse_args()

    if args.probe:
        probe(args.url, args.probe)
        return
    if args.list_json:
        list_json(args.url)
        return
    if args.extract:
        extract_one(args.url, args.extract, Path(args.out) / "visualnews_data.json")
        return

    out = Path(args.out)
    (out / "images").mkdir(parents=True, exist_ok=True)
    dj = out / "visualnews_data.json"
    if not dj.exists():
        tf = stream(args.url)
        t0 = time.time()
        for i, m in enumerate(tf):
            if m.isfile() and m.name.endswith("data.json"):
                dj.write_bytes(tf.extractfile(m).read())
                print(f"found {m.name} after {i + 1} members, {time.time() - t0:.0f}s")
                break
            if (i + 1) % 200000 == 0:
                print(f"  scanning for data.json: {i + 1} members, {time.time() - t0:.0f}s", flush=True)
        else:
            raise SystemExit("data.json not found in archive")

    vn = {str(e["id"]): e for e in json.loads(dj.read_text())}
    splits = {}
    for split, n in (("test", args.n_test), ("val", args.n_val)):
        anns = json.loads((Path(args.annotations) / f"{split}.json").read_text())["annotations"]
        splits[split] = sample(anns, n, args.seed)

    need = {}
    for picked in splits.values():
        for a in picked:
            need[norm(vn[str(a["image_id"])]["image_path"])] = None
    todo = {p for p in need if not (out / "images" / p).exists()}
    print(f"{len(need)} images needed, {len(todo)} still to extract", flush=True)

    if todo:
        tf = stream(args.url)
        t0 = time.time()
        for i, m in enumerate(tf):
            key = norm(m.name)
            if m.isfile() and key in todo:
                dest = out / "images" / key
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(tf.extractfile(m).read())
                todo.discard(key)
                if not todo:
                    print(f"all images extracted after {i + 1} members, {time.time() - t0:.0f}s")
                    break
            if (i + 1) % 200000 == 0:
                print(f"  {i + 1} members scanned, {len(todo)} images left, {time.time() - t0:.0f}s", flush=True)
        if todo:
            print(f"WARNING: {len(todo)} images not found in archive; those items are dropped")

    for split, picked in splits.items():
        rows = []
        for a in picked:
            key = norm(vn[str(a["image_id"])]["image_path"])
            img = out / "images" / key
            if not img.exists():
                continue
            rows.append({"id": f"ncp_{split}_{a['id']}_{a['image_id']}", "image_path": str(img),
                         "caption": vn[str(a["id"])]["caption"], "label": int(bool(a["falsified"]))})
        path = Path("data") / f"newsclippings_{split}.jsonl"
        path.write_text("".join(json.dumps(r) + "\n" for r in rows))
        print(f"{split}: wrote {len(rows)} records to {path}  (falsified={sum(r['label'] for r in rows)})")


if __name__ == "__main__":
    main()

Overwriting scripts/prepare_newsclippings_subset.py


In [15]:
!python -u scripts/prepare_newsclippings_subset.py --url https://www.cs.rice.edu/~vo9/visualnews/articles.tar.gz --list-json

^C
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_newsclippings_subset.py", line 140, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_newsclippings_subset.py", line 72, in main
    list_json(args.url)
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_newsclippings_subset.py", line 28, in list_json
    for i, m in enumerate(stream(url)):
                ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/tarfile.py", line 2881, in __iter__
    tarinfo = self.next()
              ^^^^^^^^^^^
  File "/usr/lib/python3.12/tarfile.py", line 2733, in next
    self.fileobj.seek(self.offset - 1)
  File "/usr/lib/python3.12/tarfile.py", line 519, in seek
    self.read(self.bufsize)
  File "/usr/lib/python3.12/tarfile.py", line 528, in read
    buf = self._read(size)
          ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/tarfile.py", line 546, in _read
    buf = self.fileobj.read(self.bufsiz

In [18]:
!find /kaggle/working -maxdepth 4 -type d -name embeddings

/kaggle/working/news_clippings/news_clippings/embeddings


In [19]:
!python scripts/clip_baseline.py newsclippings --ncp-dir /kaggle/working/news_clippings/news_clippings

val: 7024/7024 pairs with both embeddings
test: 7264/7264 pairs with both embeddings

NewsCLIPpings merged_balanced (label 1 = falsified)
CLIP zero-shot (1 - cosine)                val AUROC 0.7128 | TEST AUROC 0.7197 | acc 0.6531 | macro F1 0.6519 (threshold 0.699 from val)
(logistic probe fitted on val, so its val AUROC is optimistic)
CLIP features + logistic probe (fit on val) val AUROC 0.7419 | TEST AUROC 0.7122 | acc 0.6517 | macro F1 0.6517 (threshold 0.495 from val)


In [20]:
!python scripts/clip_baseline.py cosmos

config.json: 4.19kB [00:00, 15.2MB/s]
pytorch_model.bin: 100%|█████████████████████| 605M/605M [00:10<00:00, 58.0MB/s]
Loading weights:  29%|▎| 114/398 [00:00<00:00, 2689.00it/s, Materializing param=
Loading weights: 100%|█| 398/398 [00:00<00:00, 3554.25it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

model.safetensors:   0%|                             | 0.00/605M [00:00<?, ?B/s]
model.safetensors:   0%|                             | 0.00/605M [00:00<?, ?B/s]
model.safetensors:   1%|▎                   | 8.39M/605M [00:00<00:14, 41.9MB/s]
preprocessor_config.json: 100%|████████████████| 316/316 [00:00<00:00, 1.

In [22]:
!ls /kaggle/input/datasets/*/ | head -30

cosmos-dataset
misinfodet-predictions
mmfakebench
stage1-best-v2


In [23]:
!ls /kaggle/input/datasets/*/cosmos-dataset

cosmos_anns  images_test_clean	images_train  images_val


In [24]:
!grep -n "cosmos" configs/base.yaml

12:cosmos_dir: data/cosmos


In [25]:
!grep -n -i "path\|dir\|json\|labeled\|split" scripts/prepare_cosmos.py | head -30

2:"""Prepare COSMOS train/val (unlabeled) and test (labeled) into unified JSONL."""
4:from pathlib import Path
16:        cfg.cosmos_dir, cfg.data_dir,
20:    for split, n in counts.items():
21:        print(f"{split}: {n} records")


In [26]:
!ls /kaggle/input/datasets/*/cosmos-dataset/cosmos_anns /kaggle/input/datasets/*/cosmos-dataset/images_test_clean | head

/kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns:
test_data.json
train_data.json
val_data.json

/kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_test_clean:
test


In [27]:
!mkdir -p data/cosmos && C=$(ls -d /kaggle/input/datasets/*/cosmos-dataset); for f in $C/cosmos_anns/*.json; do ln -sfn $f data/cosmos/; done; ln -sfn $C/images_test_clean/test data/cosmos/test; ln -sfn $C/images_train/train data/cosmos/train; ln -sfn $C/images_val/val data/cosmos/val; ls -la data/cosmos

total 32
drwxr-xr-x 2 root root 4096 Oct 21 14:07 .
drwxr-xr-x 3 root root 4096 Oct 21 14:07 ..
lrwxrwxrwx 1 root root   74 Oct 21 14:07 test -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_test_clean/test
lrwxrwxrwx 1 root root   78 Oct 21 14:07 test_data.json -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/test_data.json
lrwxrwxrwx 1 root root   70 Oct 21 14:07 train -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_train/train
lrwxrwxrwx 1 root root   79 Oct 21 14:07 train_data.json -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/train_data.json
lrwxrwxrwx 1 root root   66 Oct 21 14:07 val -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/images_val/val
lrwxrwxrwx 1 root root   77 Oct 21 14:07 val_data.json -> /kaggle/input/datasets/tebogoseopa20/cosmos-dataset/cosmos_anns/val_data.json


In [28]:
!python scripts/prepare_cosmos.py --config configs/base_llava_eval.yaml

Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/prepare_cosmos.py", line 6, in <module>
    from misinfodet.data.cosmos import prepare_all
ModuleNotFoundError: No module named 'misinfodet'


In [29]:
%cd /kaggle/working/Honours-Research-Project

/kaggle/working/Honours-Research-Project


In [30]:
%env PYTHONPATH=/kaggle/working/Honours-Research-Project/src


env: PYTHONPATH=/kaggle/working/Honours-Research-Project/src


In [31]:
!python scripts/prepare_cosmos.py --config configs/base_llava_eval.yaml

[2026-10-21 14:10:17,513] INFO misinfodet.data.cosmos: COSMOS train: wrote 266028 unlabeled records to data/cosmos_train.jsonl
[2026-10-21 14:10:25,271] INFO misinfodet.data.cosmos: COSMOS val: wrote 66584 unlabeled records to data/cosmos_val.jsonl
[2026-10-21 14:10:29,182] INFO misinfodet.data.cosmos: COSMOS test: skipped 1 records with missing image files (known artifact, see docstring)
[2026-10-21 14:10:30,235] INFO misinfodet.data.cosmos: COSMOS labeled split (seed=42): 1189 train / 254 val / 256 test
train: 266028 records
val: 66584 records
labeled_train: 1189 records
labeled_val: 254 records
labeled_test: 256 records


In [32]:
!python scripts/clip_baseline.py cosmos

Loading weights: 100%|█| 398/398 [00:00<00:00, 3088.26it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/clip_baseline.py", line 212, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/clip

In [33]:
p = "scripts/clip_baseline.py"
s = open(p).read()
s = s.replace('            e = model.get_image_features(**proc(images=ims, return_tensors="pt").to(dev))\n',
              '            px = proc(images=ims, return_tensors="pt")["pixel_values"].to(dev)\n'
              '            e = model.visual_projection(model.vision_model(pixel_values=px).pooler_output)\n')
s = s.replace('            out.append(model.get_text_features(**inp).float().cpu().numpy())\n',
              '            t = model.text_projection(model.text_model(input_ids=inp["input_ids"], attention_mask=inp["attention_mask"]).pooler_output)\n'
              '            out.append(t.float().cpu().numpy())\n')
open(p, "w").write(s)
print("patched" if "visual_projection" in s and "text_projection" in s else "NOT patched")

patched


In [34]:
!python scripts/clip_baseline.py cosmos

Loading weights: 100%|█| 398/398 [00:00<00:00, 3441.04it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
  images 254/254
  images 256/256

COSMOS labelled test (label 1 = out-of-context)
image-caption gap |cos(I,C1) - cos(I,C2)|  val AUROC 0.5454 | TEST AUROC 0.5486 | acc 0.5430 | macro F1 0.5386 (threshold 

In [36]:
!mkdir -p outputs/predictions


In [37]:
!python scripts/clip_baseline.py cosmos

Loading weights: 100%|█| 398/398 [00:00<00:00, 2693.44it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
  images 254/254
  images 256/256

COSMOS labelled test (label 1 = out-of-context)
image-caption gap |cos(I,C1) - cos(I,C2)|  val AUROC 0.5454 | TEST AUROC 0.5486 | acc 0.5430 | macro F1 0.5386 (threshold 

In [39]:
!cp -n $(find /kaggle/input/datasets/*/misinfodet-predictions -name "diag_stage1_only_cosmos_scores.jsonl") outputs/predictions/

In [40]:
!python scripts/compare_runs.py --cosmos diag_stage1_only clip

Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/compare_runs.py", line 91, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/compare_runs.py", line 56, in main
    calib, evals, a, b = cosmos_items(path_a, path_b)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/scripts/compare_runs.py", line 26, in cosmos_items
    rows = list(read_jsonl(path_a))
                ^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Honours-Research-Project/src/misinfodet/utils.py", line 164, in read_jsonl
    with open(path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'diag_stage1_only'


In [41]:
!python scripts/compare_runs.py --cosmos outputs/predictions/diag_stage1_only_cosmos_scores.jsonl outputs/predictions/clip_cosmos_scores.jsonl

A = outputs/predictions/diag_stage1_only_cosmos_scores.jsonl  (threshold 0.81)
B = outputs/predictions/clip_cosmos_scores.jsonl  (threshold 0.49)
n evaluation items = 256, bootstrap iterations = 5000

AUROC     A-B = +0.3016   95% CI [+0.2274, +0.3779]   p~0.000   (significant)
Accuracy  A-B = +0.2344   95% CI [+0.1602, +0.3125]   p~0.000   (significant)


In [42]:
!python scripts/clip_baseline.py mmfakebench

Loading weights: 100%|█| 398/398 [00:00<00:00, 3330.33it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Traceback (most recent call last):
  File "/kaggle/working/Honours-Research-Project/scripts/clip_baseline.py", line 214, in <module>
    main()
  File "/kaggle/working/Honours-Research-Project/scripts/clip

In [43]:
!mkdir -p data/mmfakebench && SRC=$(ls -d /kaggle/input/datasets/*/mmfakebench); ln -sfn $SRC/MMFakeBench_test.json data/mmfakebench/; for d in $SRC/MMFakeBench_test/*; do ln -sfn $d data/mmfakebench/; done; ls data/mmfakebench

fake  MMFakeBench_test.json  real  source


In [44]:
!python scripts/prepare_mmfakebench.py --config configs/base_llava_eval.yaml

[2026-10-21 14:38:43,021] INFO misinfodet.data.mmfakebench: MMFakeBench test: wrote 10000 records to data/mmfakebench_test.jsonl
Prepared 10000 MMFakeBench records


In [45]:
!python scripts/clip_baseline.py mmfakebench

Loading weights: 100%|█| 398/398 [00:00<00:00, 3490.82it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
  images 1000/1000
  images 1000/1000

MMFakeBench sample (label 1 = any fake)
CLIP zero-shot (1 - cosine)                val AUROC 0.6409 | TEST AUROC 0.6488 | acc 0.6410 | macro F1 0.5974 (threshold 0.69

In [46]:
!cp outputs/predictions/clip_*.jsonl /kaggle/working/ && ls -la /kaggle/working/*.jsonl

-rw-r--r-- 1 root root  38614 Oct 21 15:09 /kaggle/working/clip_cosmos_scores.jsonl
-rw-r--r-- 1 root root 107524 Oct 21 15:09 /kaggle/working/clip_mmfakebench_scores.jsonl


In [47]:
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/clip_cosmos_scores.jsonl"))
display(FileLink("/kaggle/working/clip_mmfakebench_scores.jsonl"))

/kaggle/working/clip_cosmos_scores.jsonl

/kaggle/working/clip_mmfakebench_scores.jsonl

In [50]:
%%writefile scripts/fusion_calibration.py

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [55]:
%%writefile scripts/fusion_calibration.py
import argparse
import json
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PRED = Path("outputs/predictions")


def load(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    return {r["id"]: r for r in rows}


def split_arrays(d, split):
    ids = sorted((k for k, r in d.items() if r.get("split", k.split("_")[0]) == split), key=lambda k: int(k.split("_")[-1]))
    return ids, np.array([d[k]["score"] for k in ids], float), np.array([d[k]["label"] for k in ids], int)


def macro_f1(y, p):
    f = []
    for c in (0, 1):
        tp = np.sum((p == c) & (y == c))
        fp = np.sum((p == c) & (y != c))
        fn = np.sum((p != c) & (y == c))
        f.append(2 * tp / (2 * tp + fp + fn) if tp else 0.0)
    return float(np.mean(f))


def threshold(sv, yv):
    cands = np.unique(np.quantile(sv, np.linspace(0, 1, 401)))
    return max(cands, key=lambda t: macro_f1(yv, (sv >= t).astype(int)))


def bootstrap(y, a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) < 2:
            continue
        d.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
    d = np.array(d)
    return roc_auc_score(y, a) - roc_auc_score(y, b), np.percentile(d, [2.5, 97.5]), min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fusion(args):
    internal = load(PRED / f"{args.run}_cosmos_scores.jsonl")
    print(f"internal check: {args.run}_cosmos_scores.jsonl")
    for ext in args.external:
        path = PRED / f"{args.run}_{ext}_cosmos_scores.jsonl"
        if not path.exists():
            print(f"\n(skipping {path.name}: not found)")
            continue
        external = load(path)
        common = set(internal) & set(external)
        I = {k: internal[k] for k in common}
        E = {k: external[k] for k in common}
        _, iv, yv = split_arrays(I, "val")
        _, it, yt = split_arrays(I, "test")
        _, ev, _ = split_arrays(E, "val")
        _, et, _ = split_arrays(E, "test")
        print(f"\n=== external check: {path.name}  (val n={len(yv)}, test n={len(yt)}) ===")
        if len(yv) == 0:
            print("no val items in the external file; combiners cannot be fitted without leaking test data")
            continue
        Xv = np.stack([logit(iv), logit(ev)], 1)
        Xt = np.stack([logit(it), logit(et)], 1)
        clf = LogisticRegression().fit(Xv, yv)
        variants = {
            "internal only": (iv, it),
            "external only": (ev, et),
            "mean of the two": ((iv + ev) / 2, (it + et) / 2),
            "max of the two": (np.maximum(iv, ev), np.maximum(it, et)),
            "logistic fusion (fit on val)": (clf.predict_proba(Xv)[:, 1], clf.predict_proba(Xt)[:, 1]),
        }
        print(f"{'method':30s} {'AUROC':>6s} {'acc':>6s} {'mF1':>6s}   vs internal only")
        for name, (sv, st) in variants.items():
            t = threshold(sv, yv)
            p = (st >= t).astype(int)
            line = f"{name:30s} {roc_auc_score(yt, st):.3f} {np.mean(p == yt):.3f} {macro_f1(yt, p):.3f}"
            if name != "internal only":
                d, ci, pv = bootstrap(yt, st, it)
                line += f"   {d:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}] p={pv:.3f}"
            print(line)
        print(f"   logistic weights: internal={clf.coef_[0][0]:+.2f}  external={clf.coef_[0][1]:+.2f}")


def ece(p, y, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    tot, table = 0.0, []
    for b in range(bins):
        m = idx == b
        if m.any():
            conf, acc = p[m].mean(), y[m].mean()
            tot += m.mean() * abs(acc - conf)
            table.append((edges[b], edges[b + 1], int(m.sum()), conf, acc))
    return tot, table


def calibration(args):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="perfect calibration")
    print(f"{'run':28s} {'AUROC':>6s} {'ECE raw':>8s} {'Brier raw':>9s} {'mean p':>7s} {'OOC rate':>8s} {'ECE Platt':>9s} {'Brier Platt':>11s}")
    for run in args.runs:
        path = PRED / f"{run}_cosmos_scores.jsonl"
        if not path.exists():
            print(f"{run:28s} (not found)")
            continue
        d = load(path)
        _, sv, yv = split_arrays(d, "val")
        _, st, yt = split_arrays(d, "test")
        e_raw, table = ece(st, yt)
        platt = LogisticRegression().fit(logit(sv)[:, None], yv)
        pt = platt.predict_proba(logit(st)[:, None])[:, 1]
        e_cal, _ = ece(pt, yt)
        print(f"{run:28s} {roc_auc_score(yt, st):.3f} {e_raw:8.3f} {np.mean((st - yt) ** 2):9.3f} {st.mean():7.3f} {yt.mean():8.3f} "
              f"{e_cal:9.3f} {np.mean((pt - yt) ** 2):11.3f}")
        if args.table and run == args.runs[0]:
            print("   reliability (raw, test):  bin  n  mean predicted P(OOC)  observed OOC rate")
            for lo, hi, n, c, a in table:
                print(f"   [{lo:.2f},{hi:.2f})  {n:4d}  {c:.3f}  {a:.3f}")
        ax.plot([r[3] for r in table], [r[4] for r in table], "o-", label=f"{run} (ECE {e_raw:.3f})")
    ax.set_xlabel("predicted P(out-of-context)")
    ax.set_ylabel("observed out-of-context rate")
    ax.set_title("Reliability of raw verdict probabilities (COSMOS test)")
    ax.legend(fontsize=8)
    out = PRED / "reliability_cosmos.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"\nreliability diagram saved to {out}")


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("fusion")
    p.add_argument("--run", default="diag_stage1_only")
    p.add_argument("--external", nargs="+", default=["retrieval_evonly", "retrieval", "retrieval_template_only"])
    p = sub.add_parser("calibration")
    p.add_argument("--runs", nargs="+", default=["diag_stage1_only", "base_llava_eval", "stage2_eval_lmonly"])
    p.add_argument("--table", action="store_true")
    args = ap.parse_args()
    {"fusion": fusion, "calibration": calibration}[args.cmd](args)


if __name__ == "__main__":
    main()

Overwriting scripts/fusion_calibration.py


In [56]:
!mkdir -p outputs/predictions && cp -n $(find /kaggle/input/datasets/*/misinfodet-predictions -name "*_cosmos_scores.jsonl") outputs/predictions/ && ls outputs/predictions

base_llava_eval_cosmos_scores.jsonl
clip_cosmos_scores.jsonl
clip_mmfakebench_scores.jsonl
diag_stage1_only_blank_cosmos_scores.jsonl
diag_stage1_only_cosmos_scores.jsonl
diag_stage1_only_retrieval_cosmos_scores.jsonl
diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl
diag_stage1_only_retrieval_template_only_cosmos_scores.jsonl
sbert_caption_sim_cosmos_scores.jsonl
stage2_eval_lmonly_blank_cosmos_scores.jsonl
stage2_eval_lmonly_cosmos_scores.jsonl


In [57]:
!python scripts/fusion_calibration.py fusion

internal check: diag_stage1_only_cosmos_scores.jsonl

=== external check: diag_stage1_only_retrieval_evonly_cosmos_scores.jsonl  (val n=254, test n=256) ===
method                          AUROC    acc    mF1   vs internal only
internal only                  0.910 0.828 0.828
external only                  0.862 0.750 0.746   -0.048 [-0.091, -0.005] p=0.027
mean of the two                0.917 0.832 0.831   +0.008 [-0.017, +0.033] p=0.564
max of the two                 0.886 0.785 0.784   -0.024 [-0.057, +0.010] p=0.164
logistic fusion (fit on val)   0.923 0.840 0.839   +0.013 [-0.000, +0.028] p=0.052
   logistic weights: internal=+3.09  external=+0.91

=== external check: diag_stage1_only_retrieval_cosmos_scores.jsonl  (val n=254, test n=256) ===
method                          AUROC    acc    mF1   vs internal only
internal only                  0.910 0.828 0.828
external only                  0.747 0.676 0.675   -0.163 [-0.226, -0.102] p=0.000
mean of the two                0.861 0.

In [58]:
!python scripts/fusion_calibration.py calibration --table

run                           AUROC  ECE raw Brier raw  mean p OOC rate ECE Platt Brier Platt
diag_stage1_only             0.910    0.338     0.289   0.795    0.500     0.070       0.123
   reliability (raw, test):  bin  n  mean predicted P(OOC)  observed OOC rate
   [0.47,0.53)     1  0.489  0.000
   [0.53,0.60)     5  0.563  0.000
   [0.60,0.67)    11  0.640  0.000
   [0.67,0.73)    39  0.707  0.026
   [0.73,0.80)    60  0.766  0.233
   [0.80,0.87)    83  0.832  0.687
   [0.87,0.93)    57  0.885  0.982
base_llava_eval              0.874    0.341     0.327   0.839    0.500     0.083       0.147
stage2_eval_lmonly           0.893    0.348     0.327   0.844    0.500     0.064       0.135

reliability diagram saved to outputs/predictions/reliability_cosmos.png


In [59]:
!cp outputs/predictions/reliability_cosmos.png /kaggle/working/

In [60]:
!find /kaggle/input -path "*explanation*" | head -20; find /kaggle/input -name "rating*" | head

/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/explanations_cosmos_test.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack/explanations_cosmos_test.jsonl
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack/rating_images
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack/rating_sheet.csv
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack_blind
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack_blind/rating_images
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack_blind/rating_key.csv
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack_blind/rating_sheet_blind.csv
/kaggle/input/datasets/tebogoseopa20/misinfodet-predictions/rating_pack_blind/rating_sheet_rater2.csv
